# [2.4] - RLHF (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/04_[2.4]_RLHF)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part4_rlhf/2.4_RLHF_exercises.ipynb?t=20260520) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part4_rlhf/2.4_RLHF_solutions.ipynb?t=20260520)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://info-arena.github.io/ARENA_img/slack.html), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-24.png" width="350">

# Introduction

This section is designed to take you through a full implementation of RLHF (Reinforcement Learning from Human Feedback). Much of this follows on directly from the PPO implementation from yesterday, with only a few minor adjustments and new concepts. You'll (hopefully) be pleased to learn that we're disposing of OpenAI's gym environment for this final day of exercises, and instead going back to our week 1 roots with TransformerLens!

We'll start by discussing how the RL setting we've used for tasks like CartPole and Atari fits into the world of autoregressive transformer language models. We'll then go through standard parts of the PPO setup (e.g. objective function, memory buffer, rollout and learning phases) and show how to adapt them for our transformer. Finally, we'll put everything together into a `RLHFTrainer` class, and perform RLHF on our transformer!

> **Note - these exercises assume you're running on an A100 (either a virtual machine or Colab Pro+).** If you're running on machine with much less VRAM (<24GB), we recommend setting `LOW_GPU_MEM = True` below. This will switch the model to RLHF from `"gpt2-medium"` to `"gpt2-small"`,
as well as adjust some other parameters like the batch size, the number of tokens generated, and some hyperparameters.

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/wW__XFKIESc" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Content & Learning Objectives

### 1️⃣ RLHF on transformer language models

Most of the exercises today build towards the implementation of the `RLHFTrainer` class, similar to how DQN and PPO have worked these last few days.

> ##### Learning Objectives
>
> - Understand how the RL agent / action / environment paradigm works in the context of autoregressive transformer models
> - Understand how the RLHF algorithm works, and how it fits on top of PPO
> - Learn about value heads, and how they can be used to turn transformers into actor & critic networks with shared architectures
> - Write a full RLHF training loop, and use it to train your transformer with the "maximize output of periods" reward function
> - Observe and understand the instances of mode collapse that occur when training with this reward function
> - Experiment with different reward functions & training hyperparameters

### 2️⃣ LoRA

> ##### Learning Objectives
>
> - Understand the mechanism behind Low-Rank Adaptors, and how they allow for fine-tuning with less resources.
> - Implement LoRA in a transformer model.
> - Fine-tune larger models that would otherwise take too much VRAM to be possible.

### 3️⃣ GRPO LoRA

GRPO is a variant of PPO specialised for doing RLHF on LLMs. It forgoes the critic, and uses the average reward over many rollouts as a baseline instead.

> ##### Learning Objectives
>
> - Understand and implement GRPO
> - Use GRPO + LoRA together to finetune a model.

### ☆ Bonus

This section offers some suggested ways to extend the core RLHF exercises.

> ##### Learning Objectives
>  
> - Improve your RLHF implementation via techniques like differential learning rates, frozen layers, or adaptive KL penalties
> - Perform some exploratory mechanistic interpretability on RLHF'd models
> - Learn about the trlX library, which is designed to train transformers via RLHF in a way which abstracts away many of the low-level details

## Reading

- [Illustrating Reinforcement Learning from Human Feedback (RLHF)](https://huggingface.co/blog/rlhf) (~10 minutes)
    - An accessible and mostly non-technical introduction to RLHF, which discusses it in context of the full pipeline for training autoregressive transformer language models (starting with pretraining, which is what we did in the first day of last week).
- [RLHF+ChatGPT: What you must know](https://www.youtube.com/watch?v=PBH2nImUM5c) (~5 minutes)
    - The first half of this video provides a high-level overview of RLHF, discussing things like mode collapse, and relates this to the [shoggoth meme](https://i.kym-cdn.com/photos/images/original/002/546/572/bd3.png) that many of you have likely seen!
- [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/pdf/2402.03300) (~20 minutes)
    - Save reading this now until you get to the section for GRPO, and skim as required.

## Setup code

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install transformer_lens jaxtyping eindex-callum wandb

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [2]:
import os
import sys
import time
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Callable, Literal

import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from eindex import eindex
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from tabulate import tabulate
from torch import Tensor
from tqdm import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.hook_points import HookPoint

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part4_rlhf"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

from part4_rlhf import tests, tests_lora  # , tl_ext

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


MAIN = __name__ == "__main__"

# 1️⃣ RLHF on transformer language models

> ##### Learning Objectives
>
> - Understand how the RL agent / action / environment paradigm works in the context of autoregressive transformer models
> - Understand how the RLHF algorithm works, and how it fits on top of PPO
> - Learn about value heads, and how they can be used to turn transformers into actor & critic networks with shared architectures
> - Write a full RLHF training loop, and use it to train your transformer with the "maximize output of periods" reward function
> - Observe and understand the instances of mode collapse that occur when training with this reward function
> - Experiment with different reward functions & training hyperparameters

## The "transformer environment"

We'll start by discussing how we apply the reinforcement learning framework of states/actions/rewards to the setting of autoregressive language modelling. Lots of our intuitions should carry over from yesterday, it's just some of the details that have changed!

### States, actions and episodes

Our actor is an autoregressive language model. The actions $a_t$ are the tokens generated by the model (i.e. the action space is the model's vocabulary). The states $s_t$ are **the entire sequence up to that point** (not just the most recent token). In other words, given a state $s_t$ (sequence) and action $a_t$ (token generation), our new state is the concatenation which we'll denote as $s_{t+1} = [s_t \; a_t]$. For every timestep before the end of the episode, the reward is zero, and for the final timestep, the reward is given by the reward function, given the entire sequence $r_T = R(s_T)$.

Each episode is a fixed length (i.e. all our sampled outputs will have the same number of tokens generated from them). Each episode starts with an initial "prefix prompt", which is chosen before the start of training. This means that discoutning would only scale the final reward by a fixed constant, and so we don't need to worry about it here.

### Rewards and value functions

The reward $r_T$ is a function of the sequence $s_T$. Sometimes it will be a very simple function like the sum of periods `.` in the sequence, other times it'll get a bit more complicated (e.g. using a text classification model to estimate the sentiment of a sequence - we'll do this later!).

In our case, we'll only evaluate the reward at the end of the episode. This means we don't really have a concept of discount factors here - the reward only comes once, and as soon as it comes our episode terminates.

The value function $V(s_t)$ is an estimate of the expected sum of future rewards (up to the end of the episode), which in this case means it's an estimate of what the reward $r_T$ will be once we get to the end of the sequence. We'll be adding a value head to our transformer model to estimate this value function (more on this later).

> Note - a key part of RLHF is the actual gathering of and learning from human feedback, in order to train the reward function. We're not going to be doing that here, instead we'll be working with a fixed reward function. This means our implementation today is a lot more like classical reinforcement learning, and we'll be able to structure it in a way which is very similar to yesterday's PPO implementation.

### ~~Generalized~~ Advantage Estimation

We won't be using the GAE formula today for computing advantages, we'll just be directly computing it via $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$, where $a_t$ is the action which was actually taken and $Q(s_t, a_t)$ is the critic's estimate of the value function at this new state $s_{t+1} = [s_t \; a_t]$.

We can get away with this because our setup has pretty low variance when it comes to the advantage of particular actions. GAE is most helpful when it reduces variance in the advantage estimation (it does this at the cost of introducing more bias from including future value function estimates), and so it's especially useful when our environment is one with high variability when the advantage (and optimal policy) changes significantly between steps. But this doesn't really apply to us, since every action just adds a single token onto our sequence.

That said, you're welcome to experiment with the setup and try to use GAE instead! This is suggested as a bonus exercise at the end.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/transformer-rl-state.png" width="700">

## RLHF Setup

With this context in mind, we're now ready to look at the full RLHF setup we'll be using:

<img src="https://pbs.twimg.com/media/FkLOrrPWYAAiFLF.jpg:large" width="700">

Our autoregressive transformer model (we'll be using GPT2-Small) is the actor, and its value head will play the role of the critic. We follow the standard PPO setup:

- In **rollout phase**, the actor generates a bunch of sequences all starting from the prefix prompt. We compute advantage estimates using the critic network (value head) and store the experiences in memory.
- In **learning phase**, we sample from these generated experiences (i.e. from a bunch of generated sequences of different lengths, some of which might be prefixes of each other). We compute our objective function (which is the sum of the same 3 terms as yesterday) and perform a gradient step wrt it.

The only new element is the **KL prediction shift penalty**. This is a penalty we add to our overall loss function to stop the transformer from diverging too much from its initial distribution. We want to make our transformer maximize reward, but not in a way which causes it to become completely incoherent!

Note that we compute $D_{KL}(\pi_{PPO} || \pi_{base})$, not the other way around. This is because we want to penalize our new model for generating outputs which would be **extremely unlikely under the old model**, i.e. when $\pi_{PPO}$ is high and $\pi_{base}$ is low. We generally want to focus our model's output into a more concentrated version of the distribution it already has. For example in RLHF, we want to keep a low probability on completely incoherent behaviour which the original model would never have generated. But on the other hand, it's clearly fine for there to be some behaviours (e.g. offensive hate speech) which have a nontrivial probability in our base model but near-zero probability in our new model - in fact this is often desireable! For more on the intuition behind this orientation of the distributions in KL divergence, see [this post](https://www.lesswrong.com/posts/no5jDTut5Byjqb4j5/six-and-a-half-intuitions-for-kl-divergence).

<!-- An alternative perspective can be found from [this post](https://www.lesswrong.com/posts/no5jDTut5Byjqb4j5/six-and-a-half-intuitions-for-kl-divergence) - the KL divergence $D_{KL}(P || Q)$ is large when the observations $P$ give you a lot of evidence that your hypothesis $Q$ is false. We want to make sure that the original (probably coherent and sensible) model $Q$ is still a good approximation for how $P$ behaves, i.e. it shouldn't be too obvious when we observe the outputs of $P$ that they've been generated by a different model. -->

<details>
<summary>KL divergence v.s. reverse KL divergence</summary>
Assume $P$ is the true distribution, and $Q$ is the distribution we're trying to fit to $P$.

* $D_{KL}(P || Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$ blows up when $Q(x)$ is zero and $P(x)$ is positive, so we would expect that $Q$
tries to "cover" $P$ anywhere where $P(x)$ is positive. This means that minimizing $D_{KL}(\pi_{base} || \pi_{PPO})$ will cause our model to be able to do everything the base model can do, plus it can also do things out-of-distribution for the base model, which is undesirable.

* $D_{KL}(Q || P) = \sum_x Q(x) \log \frac{Q(x)}{P(x)}$ blows up when $P(x)$ is zero and $Q(x)$ is positive, so $Q$ should never assign
any probability mass to something that $P$ doesn't ($P$ "covers" $Q$), but $Q$ will instead try to cover a subset of $P$ that it fits the best.

This can be illustrated with an example. Let $P$ be a mixture of two Gaussians, and $Q \sim \mathcal{N}(\mu, \sigma^2)$ be a unimodal Gaussian (blue) parameterized by $\mu$ and $\sigma^2$. We learn parameters $\mu,\sigma^2$ that minimize both $D_{KL}(P || Q)$ and $D_{KL}(Q || P)$, and draw the resulting distribution $Q$ (here in blue), showing the expected behaviour.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/kl_diff.png" width="700">


</details>

### Summary
 
Since we're using a fixed reward function rather than training it from human feedback, our RLHF implementation looks very similar to yesterday's PPO implementation. The differences are summarized in the table below:

| |  PPO (general) | RLHF |   
|---|---|---|
| **States** | Contains partial knowledge of our environment | Sequence of tokens up to this point (and the model's internal state representation of that sequence) |
| **Actions** | Something our agent can do to change its state | Generating a new token, taking us to state $s_{t+1} = [s_t \; a_t]$ |
| **Rewards** | A function of the state, which is computed after each new state is reached | A function of the sequence, can be computed after each new token but we'll just compute it once at the end of the sequence |
| **Multiple steps in parallel?** | Yes, we used `SyncVectorEnv` to parallelize the rollout phase | Yes, we'll pass batches of sequences into the transformer model, generating multiple new tokens at once |
| **Actor & critic networks** | Architectures can be shared (e.g. for Atari) or disjoint (e.g. for CartPole) | Actor is a transformer model, critic is a value head (so most architecture is shared) |
| **Advantage estimation** | Use GAE with discount factor $\lambda$ | Often uses GAE, but we'll just use simple next-step difference $V(s_{t+1}) - V(s_t)$ |
| **Anything extra?** |  | KL penalty on the new policy wrt the baseline policy |

## RLHF training args

Now that you have a rough idea of how our implementation differs from PPO, we'll give you the `RLHFArgs` class and highlight the differences between this and the `PPOArgs` class from yesterday (mostly it's quite similar).

- We're now using `total_phases` to control how long our training lasts for, rather than using `total_timesteps`. This makes more sense for us, because the total number of timesteps (= number of actions we take = number of tokens we generate) will vary depending on the length of the sequences we generate.
- We've removed the arguments `gamma` and `gae_lambda` for computing the advantage function, since as discussed we'll be computing the advantage in a simpler and more direct way (you'll do this in the next exercise).
- We've added the following arguments related to the base model & text sampling:
    - `base_model`, for specifying different base models (default is `"gpt2-small"`)
    - `gen_len`, the length of the sequences we generate.
    - `temperature` and `top_k`, for controlling the sampling temperature of our sequences.
    - `prefix`, the string we use to generate all samples.
- As well as the following extra RLHF-specific arguments:
    - `kl_coef`, for controlling the strength of the KL prediction shift penalty.
    - `reward_fn`, for the reward function we use.
    - `normalize_reward`, for whether we normalize the reward (this won't always be necessary).
- We've also added two learning rates, since it makes sense to have a different learning rate for our value head and the rest of the model (more on this later!).

In [3]:
# Set default parameters for low GPU memory usage, change if you have more GPU memory

LOW_GPU_MEM = False
BASE_MODEL = "gpt2-small" if LOW_GPU_MEM else "gpt2-medium"
RUN_BASE_RLHF = True

In [4]:
@dataclass
class RLHFArgs:
    # Basic / global
    seed: int = 1

    # Wandb / logging
    use_wandb: bool = False
    wandb_project_name: str = "RLHF"
    wandb_entity: str | None = None

    # Duration of different phases
    total_phases: int = 100
    batch_size: int = 128
    num_minibatches: int = 4
    batches_per_learning_phase: int = 2

    # Optimization hyperparameters
    base_lr: float = 2e-5
    head_lr: float = 5e-4
    max_grad_norm: float = 1.0
    warmup_steps: int = 20
    final_scale: float = 0.1

    # Computing other PPO loss functions
    clip_coef: float = 0.2
    vf_coef: float = 0.15
    ent_coef: float = 0.001

    # Base model & sampling arguments
    base_model: str = BASE_MODEL
    gen_len: int = 30
    temperature: float = 1.0
    top_k: int = 10
    prefix: str = "This is"
    prepend_bos: bool = True

    # RLHF-specific arguments
    kl_coef: float = 2.5
    reward_fn: Callable = lambda x: 0.0
    normalize_reward: bool = True

    def __post_init__(self):
        assert self.total_phases > self.warmup_steps, "total_phases must be greater than warmup_steps"
        assert self.batch_size % self.num_minibatches == 0, "batch_size should be divisible by num_minibatches"
        self.minibatch_size = self.batch_size // self.num_minibatches

## Value head

If you worked on the Atari exercises yesterday, then you'l be used to the idea of having shared architecture between our policy and value networks. Intuitively, this is because both networks need to learn some kind of high-level encoding of the important variables in the environment - they just do different things with this encoding.

This leads to the idea of a **value head**. A value head is basically just a simple classifier model which we stick to one of the policy network's internal activations. You can think of this as a kind of feature extraction. When it comes to transformer models, we usually attach our value head to **the value of the residual stream at the very last layer, after layernorm but before unembedding**. Recall the key idea of **residual stream as output accumulation** - by the very last layer, it contains the most context about the overall sequence.\*

\*Technically this might not always be true, since there is some evidence that components of a transformer erase information in order to write different information to the residual stream. However, in practice we usually find that the residual stream at the last layer is the most useful for downstream tasks.

How do we implement this? Before you read further down, try to think about how you might implement this yourself, i.e. how you could extend the functionality of your `HookedTransformer` model by adding a value head, without completely rewriting the `HookedTransformer` architecture.

<details>
<summary>Hint</summary>

Think about using hook functions.

</details>

<details>
<summary>Answer</summary>

One method would be to directly edit the model by replacing its modules with different ones. But this is a bit awkward, because we have to also change modules which are downstream of the value head to make sure that they're only taking the residual stream as input (not the value head's output), etc.

A different method, which is what we'll be using in these exercises, is to use **hook functions**. We can attach a hook function to the residual stream at the final layer, and have it apply our value head to the residual stream values & store the output externally. Then we can use `model.run_with_hooks` to get our logits like normal, and fetch our value estimate from the external storage object.

We're used to using hook functions during inference mode to perform causal interventions or compute statistical functions of our activations, but they can also be used during training mode to perform computations which are part of the autograd's computational graph.

</details>

### Exercise - implement `HookedTransformerWithValueHead`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-25 minutes on this exercise.
> ```

Here is a diagram of your implementation.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/value-head-3.png" width="600">

* Define the class method `.from_pretrained` to call the parent class's `.from_pretrained` method, and then afterwards define the value head `self.value_head`.

* We have an extra argument `use_value_head`. If it is false, just let `model.value_head = None`. We do this so we can reuse this class for the GRPO section.

* Rewrite the `forward` method so that it outputs both the logits from a forward pass *and* the output of the value head.

The easiest and most direct way to get the output of the value head would be to **add a hook to the residual stream before the unembedding matrix, which computes the output of the value head and stores it externally (or as a class attribute).** You can review the material from section 1.2 if you don't remember how to use hooks, and you can refer to the diagram on the [reference page](https://arena-chapter1-transformer-interp.streamlit.app) (find it on the left hand sidebar) for how to get the correct hook name.

<details>
<summary> Why do we need to add the hook after the layernorm? </summary>

The answer is that the residual stream can often [grow in magnitude over time](https://www.lesswrong.com/posts/8mizBCm3dyc432nK8/residual-stream-norms-grow-exponentially-over-the-forward). Our rewards will be normalized (see later exercise), and so we want to make sure the outputs of our value head (which are estimates of the reward) also start off normalized.
</details>

In [5]:
class HookedTransformerWithValueHead(HookedTransformer):
    """
    Defines a GPT model with a value head (the latter taking the last hidden state as input, post-layernorm).

    The value head is a simple MLP with one hidden layer, and scalar output:

        Linear(d_model -> 4*d_model)
        ReLU
        Linear(4*d_model -> 1)

    All linear layers have biases.
    """

    value_head: nn.Sequential
    value_head_output: Float[Tensor, "batch seq"]
    value_head_hook: list[tuple[str, Callable]]

    @classmethod
    def from_pretrained(cls, *args, use_value_head=True, **kwargs):
        model = super(HookedTransformerWithValueHead, cls).from_pretrained(*args, **kwargs)
        model.value_head_hook = ("ln_final.hook_normalized", model.run_value_head)

        if use_value_head:
            model.value_head = nn.Sequential(
                nn.Linear(model.cfg.d_model, 4 * model.cfg.d_model),
                nn.ReLU(),
                nn.Linear(4 * model.cfg.d_model, 1)
            )
        else:
            model.value_head = None
        return model

    @property
    def fwd_hooks(self):
        return [self.value_head_hook]

    def get_base_model_trainable_params(self):
        return (p for name, p in self.named_parameters() if "value_head" not in name)

    def get_value_head_params(self):
        return self.value_head.parameters()

    def run_value_head(self, resid_post: Float[Tensor, "batch seq d_model"], hook: HookPoint):
        if self.value_head is not None:
            self.value_head_output = self.value_head(resid_post).squeeze() # (batch, seq)

    def forward_with_value_head(
        self,
        input_ids: Int[Tensor, "batch seq"],
        **kwargs,
    ) -> tuple[Float[Tensor, "batch seq d_vocab"], Int[Tensor, "batch seq"]]:
        self.value_head_output = None
        logits = self.run_with_hooks(
            input_ids,
            return_type="logits",
            fwd_hooks=self.fwd_hooks
        )
        return logits, self.value_head_output


# Define a reference model (we'll use this during RLHF)
model = HookedTransformerWithValueHead.from_pretrained("pythia-14m", use_value_head=True).to(device)
tests.test_transformer_with_value_head(model)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-14m into HookedTransformer
Moving model to device:  cuda
Layer (type:depth-idx)                   Param #
Sequential                               --
├─Linear: 1-1                            66,048
├─ReLU: 1-2                              --
├─Linear: 1-3                            513
Total params: 66,561
Trainable params: 66,561
Non-trainable params: 0
All tests for `TransformerWithValueHead` passed!


<details>
<summary>Solution</summary>

We do this by storing the value head output as a property before returning it.

```python
class HookedTransformerWithValueHead(HookedTransformer):
    """
    Defines a GPT model with a value head (the latter taking the last hidden state as input, post-layernorm).

    The value head is a simple MLP with one hidden layer, and scalar output:

        Linear(d_model -> 4*d_model)
        ReLU
        Linear(4*d_model -> 1)

    All linear layers have biases.
    """

    value_head: nn.Sequential
    value_head_output: Float[Tensor, "batch seq"]
    value_head_hook: list[tuple[str, Callable]]

    @classmethod
    def from_pretrained(cls, *args, use_value_head=True, **kwargs):
        model = super(HookedTransformerWithValueHead, cls).from_pretrained(*args, **kwargs)
        model.value_head_hook = ("ln_final.hook_normalized", model.run_value_head)

        if use_value_head:
            model.value_head = nn.Sequential(
                nn.Linear(model.cfg.d_model, 4 * model.cfg.d_model), nn.ReLU(), nn.Linear(4 * model.cfg.d_model, 1)
            )
        else:
            model.value_head = None
        return model

    @property
    def fwd_hooks(self):
        return [self.value_head_hook]

    def get_base_model_trainable_params(self):
        return (p for name, p in self.named_parameters() if "value_head" not in name)

    def get_value_head_params(self):
        return self.value_head.parameters()

    def run_value_head(self, resid_post: Float[Tensor, "batch seq d_model"], hook: HookPoint):
        self.value_head_output = self.value_head(resid_post).squeeze(-1)

    def forward_with_value_head(
        self,
        input_ids: Int[Tensor, "batch seq"],
        **kwargs,
    ) -> tuple[Float[Tensor, "batch seq d_vocab"], Int[Tensor, "batch seq"]]:
        self.value_head_output = None

        logits = self.run_with_hooks(
            input_ids,
            return_type="logits",
            fwd_hooks=self.fwd_hooks,
        )

        return logits, self.value_head_output


# Define a reference model (we'll use this during RLHF)
model = HookedTransformerWithValueHead.from_pretrained("pythia-14m", use_value_head=True).to(device)
tests.test_transformer_with_value_head(model)
```

</details>

## Sampling from a transformer

If you didn't go through the sampling exercises during the first day of last week, you might want to go back to them and work through the first few of them (this is not essential). Otherwise, here's a quick refresher:

<details>
<summary>Sampling methods</summary>

- The simplest form of sampling is **greedy sampling**, where we autoregressively generate text by always choosing the most likely token at each step (i.e. argmaxing over logits), appending this to our sequence, and continuing.
- Most other forms of sampling are non-deterministic, i.e. they involve randomness. The most basic form of random sampling is choosing the next token according to the model's logit distribution.
- Other common refinements of this basic method are:
    - **Top-k sampling**, where we only consider the top-k most likely tokens at each step, and choose from these according to the model's logit distribution.
    - **Top-p sampling** (also called **nucleus sampling**), where we only consider the most likely tokens that have cumulative probability at least $p$ at each step, and choose from these according to the model's logit distribution.
</details>

We've provided the model sampling code for you below, because there are a few non-obvious things to consider that are specific to our current situation. Make sure you completely understand this function before moving on to the next section.

We'll highlight a few things about this function:

- `generate` is the standard method to autoregressively generate text. This works for TransformerLens slightly differently than for HuggingFace models (TransformerLens isn't primarily designed for text generation). In particular (at time of writing), it doesn't have features to efficiently generate multiple outputs for a single completion by using key-value caching. 
So rather than passing an argument into `generate` telling the model to generate `batch_size` outputs, we've instead just repeated `input_ids` multiple times across the batch dimension. This may sound a bit wasteful since we're repeating computation on the input sequence, but it's not a big problem because the input sequences we'll be using are usually very short. We would only expect to see a significant slowdown when the prompt was very long, and the generations very short (recall that since the prompt is run in parallel, but autoregressive sampling is sequential, *most* of the wall-time is spent waiting for the model to generate the next token).

- We've used `stop_at_eos=False`, to make sure that the model generates the full `gen_length` tokens rather than stopping early.

In [6]:
@t.no_grad()
def get_samples(
    model: HookedTransformer,
    prompt: str,
    batch_size: int,
    gen_len: int = 15,
    temperature: float = 0.8,
    top_k: int = 15,
    prepend_bos: bool = True,
    **kwargs,
) -> tuple[Int[Tensor, "batch seq"], list[str]]:
    """
    Generates samples from the model, which will be fed into the reward model and evaluated.

    Inputs:
        model: the transformer to generate samples from
        prompt: the initial prompt fed into the model
        batch_size: the number of samples to generate
        gen_len: the length of the generated samples (i.e. the number of *new* tokens to generate)
        temperature: the temp of the sampling distribution (higher means more random completions)
        top_k: the topk parameter of sampling (higher means a wider variety of possible completions)

    Returns:
        sample_ids: the token ids of the generated samples (including initial prompt)
        samples: the generated samples (including initial prompt)
    """

    # Convert our prompt into tokens
    input_ids = model.to_tokens(prompt, prepend_bos=prepend_bos)
    input_ids = einops.repeat(input_ids, "1 seq -> batch seq", batch=batch_size)

    # Generate samples
    output_ids = model.generate(
        input_ids,
        max_new_tokens=gen_len,
        stop_at_eos=False,
        temperature=temperature,
        top_k=top_k,
        **kwargs,
    )
    samples = model.to_string(output_ids)

    return output_ids.clone(), samples

Here's some example use of this function. You may wish to set `use_past_kv_cache=False` (default `True`) to see how much of a difference it makes, and `verbose=True` if you want a progress bar while generating tokens.

In [7]:
model = HookedTransformerWithValueHead.from_pretrained(BASE_MODEL).to(device)

sample_ids, samples = get_samples(
    model,
    prompt="So long, and thanks for all the",
    batch_size=5,
    gen_len=15,
    temperature=0.8,
    top_k=15,
    prepend_bos=False,
    verbose=True,
    use_past_kv_cache=True,
)

table = Table("Token IDs", "Samples", title="Demo of `sample` function", show_lines=True)
for ids, sample in zip(sample_ids, samples):
    table.add_row(str(ids.tolist()), repr(sample))

rprint(table)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda


  0%|          | 0/15 [00:00<?, ?it/s]

                                             Demo of `sample` function                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Token IDs                                              ┃ Samples                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ [2396, 890, 11, 290, 5176, 329, 477, 262, 14081, 9846, │ 'So long, and thanks for all the lovely                │
│ 0, 50256, 464, 1708, 318, 257, 8319, 1281, 416, 1583,  │ memories!<|endoftext|>The following is a guest post by │
│ 13, 3899, 402]                                         │ Dr. Michael G'                                         │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ [2396, 890, 11, 290, 5176, 329, 477, 262, 1104, 3228,  │ 'So long, and thanks for all the                       │
│ 198, 198, 12, 50, 2788, 50256, 32, 4744, 1644, 3818,   │ support!!\n\n-Seth<|endoftext|>A Florida police        │
│ 508, 6848, 284]                                        │ officer who admitted to'                               │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ [2396, 890, 11, 290, 5176, 329, 477, 262, 1049, 670,   │ 'So long, and thanks for all the great work, you are   │
│ 11, 345, 389, 4988, 20886, 262, 2831, 526, 198, 198,   │ truly inspiring the industry."\n\nThe project aims'    │
│ 464, 1628, 12031]                                      │                                                        │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ [2396, 890, 11, 290, 5176, 329, 477, 262, 1104, 11,    │ "So long, and thanks for all the support, and if you'd │
│ 290, 611, 345, 1549, 588, 284, 766, 617, 517, 286,     │ like to see some more of our amazing work"             │
│ 674, 4998, 670]                                        │                                                        │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ [2396, 890, 11, 290, 5176, 329, 477, 262, 5916, 0,     │ 'So long, and thanks for all the fish!\n\nThe fish     │
│ 198, 198, 464, 5916, 547, 1049, 0, 383, 5916, 29187,   │ were great! The fish tasted great, the'                │
│ 1049, 11, 262]                                         │                                                        │
└────────────────────────────────────────────────────────┴────────────────────────────────────────────────────────┘

The `**kwargs` argument is passed along to `generate`, so as a reference you may wish to check the docstring, or the [source code](https://github.com/TransformerLensOrg/TransformerLens/blob/f103debd1084cd79969164ac98ed9059a86354bc/transformer_lens/HookedTransformer.py#L2031) of the `generate` method. As a reminder, at time of writing we are using version `2.11.0` of TransformerLens in case you're digging deeper into the source code.

<details>
<summary><code>generate</code> docstring</summary>

```python
@torch.inference_mode()
    def generate(
        self,
        input: Union[str, Float[torch.Tensor, "batch pos"]] = "",
        max_new_tokens: int = 10,
        stop_at_eos: bool = True,
        eos_token_id: int | None = None,
        do_sample: bool = True,
        top_k: int | None = None,
        top_p: float | None = None,
        temperature: float = 1.0,
        freq_penalty: float = 0.0,
        use_past_kv_cache: bool = True,
        prepend_bos: bool | None = USE_DEFAULT_VALUE,
        padding_side: Literal["left", "right"] | None = USE_DEFAULT_VALUE,
        return_type: str | None = "input",
        verbose: bool = True,
    ) -> Union[Int[torch.Tensor, "batch pos_plus_new_tokens"], str]:
        """Sample Tokens from the Model.

        Sample tokens from the model until the model outputs eos_token or max_new_tokens is reached.

        To avoid fiddling with ragged tensors, if we input a batch of text and some sequences finish
        (by producing an EOT token), we keep running the model on the entire batch, but throw away
        the output for a finished sequence and just keep adding EOTs to pad.

        This supports entering a single string, but not a list of strings - if the strings don't
        tokenize to exactly the same length, this gets messy. If that functionality is needed,
        convert them to a batch of tokens and input that instead.

        Args:
            input (Union[str, Int[torch.Tensor, "batch pos"])]): Either a batch of tokens ([batch,
                pos]) or a text string (this will be converted to a batch of tokens with batch size
                1).
            max_new_tokens (int): Maximum number of tokens to generate.
            stop_at_eos (bool): If True, stop generating tokens when the model outputs eos_token.
            eos_token_id (int | Sequence[int] | None): The token ID to use for end
                of sentence. If None, use the tokenizer's eos_token_id - required if using
                stop_at_eos. It's also possible to provide a list of token IDs (not just the
                eos_token_id), in which case the generation will stop when any of them are output
                (useful e.g. for stable_lm).
            do_sample (bool): If True, sample from the model's output distribution. Otherwise, use
                greedy search (take the max logit each time).
            top_k (int | None): Number of tokens to sample from. If None, sample from all tokens.
            top_p (float | None): Probability mass to sample from. If 1.0, sample from all tokens. If <1.0,
                we take the top tokens with cumulative probability >= top_p.
            temperature (float): Temperature for sampling. Higher values will make the model more
                random (limit of temp -> 0 is just taking the top token, limit of temp -> inf is
                sampling from a uniform distribution).
            freq_penalty (float): Frequency penalty for sampling - how much to penalise previous
                tokens. Higher values will make the model more random.
            use_past_kv_cache (bool): If True, create and use cache to speed up generation.
            prepend_bos (bool, optional): Overrides self.cfg.default_prepend_bos. Whether to prepend
                the BOS token to the input (applicable when input is a string). Defaults to None,
                implying usage of self.cfg.default_prepend_bos (default is True unless specified
                otherwise). Pass True or False to override the default.
            padding_side (Literal["left", "right"] | None, optional): Overrides
                self.tokenizer.padding_side. Specifies which side to pad when tokenizing multiple
                strings of different lengths.
            return_type (str | None): The type of the output to return - either a string (str),
                a tensor of tokens (tensor) or whatever the format of the input was (input).
            verbose (bool): If True, show tqdm progress bars for generation.

        Returns:
            outputs (torch.Tensor): [batch, pos + max_new_tokens], generated sequence of new tokens
                (by default returns same type as input).
        """
```
</details>

### Exercise - implement `reward_fn_char_count`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend 5-10 minutes on this exercise.
> ```

We'll start with a very basic reward function: counting the total number of periods in the sequence.

An interesting thing to note about this reward function - it counts over all characters, but the episode length is defined in terms of tokens. This means that theoretically our model could reward hack by outputting tokens with more than one `.` character. This particular model's vocabulary happens to include the token `'.' * 64`, (token `23193`) so rewards would be through the roof if this was ever generated! However, remember that RL is about performing actions, getting feedback on those actions, and using that feedback to influence your policy. The token `'.' * 64` is so unlikely to ever be generated that it'll probably never be positively reinforced, and we avoid this problem.

If we were worried about this, we could instead have the reward be the number of tokens that contain at least one `.`, or normalize over character count, or something similar. For now, this is jsut an easy reward function that we can use to quickly verify that our RLHF trainer is working.

In [8]:
def reward_fn_char_count(generated_sample: list[str], char: str = ".") -> Float[Tensor, " batch"]:
    """
    Reward function (counting number of instances of a particular character), evaluated on the
    generated samples. The return type should be a tensor of floats.
    """
    return t.tensor([sample.count(char) for sample in generated_sample], dtype=t.float32, device=device)


# Test your reward function
A = "This is a test."
B = "......"
C = "Whatever"

t.testing.assert_close(reward_fn_char_count([A]), t.tensor([1.0], device=device))
t.testing.assert_close(reward_fn_char_count([A, B, C]), t.tensor([1.0, 6.0, 0.0], device=device))
t.testing.assert_close(reward_fn_char_count([A], " "), t.tensor([3.0], device=device))
print("All tests for `reward_fn_char_count` passed!")

All tests for `reward_fn_char_count` passed!


<details><summary>Solution</summary>

```python
def reward_fn_char_count(generated_sample: list[str], char: str = ".") -> Float[Tensor, " batch"]:
    """
    Reward function (counting number of instances of a particular character), evaluated on the
    generated samples. The return type should be a tensor of floats.
    """
    return t.tensor([item.count(char) for item in generated_sample], device=device, dtype=t.float)
```
</details>

### Exercise - brainstorm your reward function

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend ~5 minutes on this exercise.
> ```

Take 5 minutes (on your own or with a partner) to brainstorm how the model might be able to maximize the output of periods in ways which don't produce incoherent output (e.g. collapsing into only outputting periods). Remember we have a KL penalty with the reference model, meaning the model is penalized for producing outputs which would be very unlikely under the original model. What ideas can you come up with? When you train your model and observe the output, you should come back here and see how many of the period-maximizing behaviours you predicted actually occur.

This exercise is a great way to start thinking about the effects of different reward functions - although it's only a toy example, it still illustrates the important alignment concept that the behaviour induced by certain reward functions might not always be what you expect!

<details>
<summary>Spoiler - which behaviours will your model pick up?</summary>

The strategies adopted by the model very a lot depending on the prefix string, also thanks to mode collapse it will often find one of these behaviours and entirely ignore the others.

Some common strategies include:

- Shorter sentences
- Repeating `U.S.` or `U.S.A.` (using the prefix prompt `"There is"`, this seems to be by far the most common strategy)
- Library versions e.g. `Python 2.7.12` or `the 2.6.0.2 release`
- Names with initials e.g. `C. S. Lewis` or titles e.g. `Dr.` and `PhD.`
- Abbreviations e.g. `Data-R.A.R. series` or `"L.A. Times"`
- Decimals in numbers e.g. `9.5cm x 7.5 cm`
- Ellipses e.g. `the man . . . the woman . . .`

</details>

### Exercise - implement `normalize_reward`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend ~5 minutes on this exercise.
> ```

Following advice from Ziegler el al. (2019), it's important to normalize the reward function over each batch (i.e. subtract mean and divide by std dev). We've been able to get away with not doing this so far because our reward functions were usually nicely bounded, e.g. the reward was always zero or one in cartpole (and even in our reward shaping it was still in the zero-one range). But if we're working with reward functions that could be much higher variance such as the number of periods in a generated sequence, then we should normalize.

Note - we're not super strict about this function; the denominator being `std + eps` or `(var + eps).sqrt()` are both fine.

In [9]:
def normalize_reward(reward: Float[Tensor, " batch"], eps=1e-5) -> Float[Tensor, " batch"]:
    """
    Normalizes the reward function values over the batch of sequences.
    """
    return (reward - reward.mean()) / (reward.std() + eps)


tests.test_normalize_reward(normalize_reward)

All tests for `normalize_reward` passed!


<details><summary>Solution</summary>

```python
def normalize_reward(reward: Float[Tensor, " batch"], eps=1e-5) -> Float[Tensor, " batch"]:
    """
    Normalizes the reward function values over the batch of sequences.
    """
    return (reward - reward.mean()) / (reward.std() + eps)
```
</details>

### Exercise - implement `get_advantages`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

As we discussed earlier, your advantage function doesn't need to use GAE like yesterday. Instead, we'll base our estimates on the simple formula:

$$
A(s_t, a_t) = Q(s_t, a_t) - V(s_t)
$$

In place of $Q(s_t, a_t)$ we'll use the **one-step Q estimates**, i.e. our value function estimates after taking action $a_t$ at step $s_t$, meaning we're at new state $s_{t+1} = [s_t \; a_t]$. If $t < T$ (i.e. we're before the final sequence position) then the one-step Q estimates just equal the value function estimates $V(s_{t+1})$, but if $t=T$ then we can just use the known reward $r_t$ for the whole sequence (e.g. in our case that's the number of periods in the generated sequence).

The diagram below should help explain things. Note that the output should have shape `[minibatch_size, gen_length]` where `gen_length` is defined as `seq_len - prefix_len` i.e. the number of tokens our model generated. See the diagram below to help illustrate things, and make sure you slice your tensors carefully to match the diagram!

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/rlhf-advantages-2.png" width="900">

In [10]:
@t.no_grad()
def compute_advantages(
    values: Float[Tensor, " minibatch_size seq_len"],
    rewards: Float[Tensor, " minibatch_size"],
    prefix_len: int,
) -> Float[Tensor, " minibatch_size gen_len"]:
    """
    Computes the advantages for the PPO loss function, i.e. A_pi(s, a) = Q_pi(s, a) - V_pi(s).

    In this formula we replace Q(s, a) with the 1-step Q estimates, and V(s) with the 0-step value estimates.

    Inputs:
        values:
            the value estimates for each token in the generated sequence
        rewards:
            the rewards for the entire generated sequence
        prefix_len:
            the length of the prefix (i.e. the length of the initial prompt)

    Returns:
        advantages:
            the advantages for each token in the generated sequence (not the entire sequence)
    """
    q_estimates = t.cat((values[:,prefix_len:-1], rewards[:, None]), dim=1)
    return q_estimates - values[:,prefix_len-1:-1]


tests.test_compute_advantages(compute_advantages)

All tests in `test_compute_advantages` passed!


<details><summary>Solution</summary>

```python
@t.no_grad()
def compute_advantages(
    values: Float[Tensor, " minibatch_size seq_len"],
    rewards: Float[Tensor, " minibatch_size"],
    prefix_len: int,
) -> Float[Tensor, " minibatch_size gen_len"]:
    """
    Computes the advantages for the PPO loss function, i.e. A_pi(s, a) = Q_pi(s, a) - V_pi(s).

    In this formula we replace Q(s, a) with the 1-step Q estimates, and V(s) with the 0-step value estimates.

    Inputs:
        values:
            the value estimates for each token in the generated sequence
        rewards:
            the rewards for the entire generated sequence
        prefix_len:
            the length of the prefix (i.e. the length of the initial prompt)

    Returns:
        advantages:
            the advantages for each token in the generated sequence (not the entire sequence)
    """
    # (see diagram) stack values [3, 4, 5, 6] and rewards [7,] to get the first term in our calculation of advantages
    one_step_q_est = t.cat([values[:, prefix_len:-1], rewards[:, None]], dim=-1)

    # (see diagram) slice values [2, 3, 4, 5, 6] to get our zero-step value estimates
    zero_step_value_est = values[:, prefix_len - 1 : -1]

    advantages = one_step_q_est - zero_step_value_est
    return advantages
```
</details>

## Memory

We've given you an implementation of the `ReplayMemory` and `ReplayMinibatch` classes.

Some notes on how `ReplayMinibatch` differs from the PPO implementation, mostly in ways which make it strictly simpler:

- We don't need to store `actions` any more, because the actions (tokens generated) are in contained within the sequences themselves.
- We don't need to store `dones` any more, because all our sequences last for exactly `gen_length` steps.
- We need to store `ref_logits`, which are used to compute the KL penalty with respect to our reference model.

Some notes on how `ReplayMemory` differs from the PPO implementation, again mostly making it simpler:

- We don't have multiple environments to flatten over, which cuts down a lot of our previous boilerplate code.
- We won't use `add` to add experience data one by one, intead we'll add it all at once.
- Many of the tensors below have shape `(batch_size, gen_len)` not `(batch_size, seq_len)`, because we only care about their values for the generated tokens, not the prefix tokens (only the generated tokens correspond to actual actions our model took).

<details>
<summary>A note on <code>returns</code>, and how this relates to DQN (optional)</summary>

Note that because we're using simple 1-step advantage estimation rather than GAE, our `returns` are just equivalent to the next-step estimates of our value function (except for `returns[:, -1]` which equals our end-of-sequence rewards). 

Recall from our discussion in PPO yesterday that the `returns` are used in the value function loss which plays a similar role to the DQN loss (of bringing the value estimates in line with the next-step value estimates). This parallel between the DQN loss and value function loss is even clearer here:

- DQN loss was the squared difference between current Q-value $Q_\theta(s_t, a_t)$ and the time-discounted next step Q-values for the target network $\theta_\text{target}$, the role was to improve $Q_\theta$ estimates
- Here, the value function loss reduces to the squared difference between the current value estimate $V_\theta(s_t)$ and the next-step value estimate $V_{\theta_\text{old}}(s_{t+1})$ computed during rollout, the role is to improve $V_\theta$ estimates

Obviously the formulas look different here becaause we have no discount ($\gamma = 1$) and we also have no rewards except at the final step ($r_t = 0 \; \forall t < T$), but the idea is fundamentally the same.

</details>

In [11]:
@dataclass
class ReplayMinibatch:
    """
    Samples from the replay memory.
    """

    sample_ids: Float[Tensor, " minibatch_size seq_len"]
    logprobs: Float[Tensor, " minibatch_size gen_len"]
    advantages: Float[Tensor, " minibatch_size gen_len"]
    returns: Float[Tensor, " minibatch_size gen_len"]
    ref_logits: Float[Tensor, " minibatch_size seq_len d_vocab"]


class ReplayMemory:
    def __init__(
        self,
        args: RLHFArgs,
        sample_ids: Float[Tensor, " batch_size seq_len"],
        logprobs: Float[Tensor, " batch_size gen_len"],
        advantages: Float[Tensor, " batch_size gen_len"],
        values: Float[Tensor, " batch_size seq_len"],
        ref_logits: Float[Tensor, " batch_size seq_len d_vocab"],
    ):
        """
        Initializes the replay memory, with all the data generated from the rollout phase at once.

        The advantages are (batch_size, gen_len) because we only compute advantages for the generated
        tokens. The other tensors, except logprobs, uses seq_len instead of gen_len because they are
        computed for all tokens.
        """

        assert ref_logits.ndim == 3
        assert ref_logits.shape[0] == args.batch_size
        assert sample_ids.shape == values.shape == ref_logits.shape[:2]
        assert advantages.shape == logprobs.shape == (args.batch_size, args.gen_len)

        self.args = args
        self.sample_ids = sample_ids
        self.logprobs = logprobs
        self.advantages = advantages
        self.values = values
        self.ref_logits = ref_logits

    def get_minibatches(self) -> list[ReplayMinibatch]:
        """
        Generates a list of minibatches by randomly sampling from the replay memory. Each sequence
        appears exactly `batches_per_learning_phase` times in total.
        """
        minibatches = []

        returns = self.advantages + self.values[:, -self.args.gen_len - 1 : -1]

        for _ in range(self.args.batches_per_learning_phase):
            for indices in t.randperm(self.args.batch_size).reshape(self.args.num_minibatches, -1):
                minibatches.append(
                    ReplayMinibatch(
                        sample_ids=self.sample_ids[indices],
                        logprobs=self.logprobs[indices],
                        advantages=self.advantages[indices],
                        returns=returns[indices],
                        ref_logits=self.ref_logits[indices],
                    )
                )

        return minibatches

## RLHF Agent?

If we were matching our implementation to our PPO implementation yesterday, this is where we'd define an `RLHFAgent` class. This class would have the role of:

- Managing interactions between the agent and the environment
- Sequentially taking steps in the environment and storing these steps as experience tuples in `ReplayMemory`

However, we're not going to do this here because it's not a useful abstraction in our case - there's no clear separation between our agent and our environment like there was yesterday. Instead, most of the extra logic in `play_step` (i.e. generating tokens and storing the associated experiences in replay memory) will be handled later in the `rollout_phase` method of your `RLHFTrainer` class.

## Objective function

### Exercise - implement `calc_kl_penalty`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Now, you'll implement the KL penalty function. As discussed, the purpose of this function is to make sure your new model doesn't diverge too much from the old model. We'll be using the KL divergence between the old and new models' logit distributions.

The formula for KL divergence of two distributions, $D_{\text{KL}}(P || Q)$, is $\sum_i P_i \log (P_i / Q_i)$. Recall that we want our new logits to be $P$ and reference logits to be $Q$ (because this penalizes our new model for generating outputs which would be very unlikely under the original reference model).

A few other tips / notes about this implementation:

- We only pass `logits` and `ref_logits` for the generated tokens
    - This is because we don't care about the model's logits for prefix tokens, since it's not in control of them
- You should pay attention to **numerical stability** when calculating KL div
    - This means for example you shouldn't take `softmax` to get probabilities _then_ `log` to get logits, since taking the log of very small numbers is unstable
    - You should instead use something like `log_softmax` to get logprobs then `exp` to get probabilities, which works since `log_softmax` is stable (it subtracts a constant from all the logits so they're not all extremely negative) and `exp` of a negative number is stable
- You should **sum over the `d_vocab`** dimension, but take the **mean over batch & pos** dimensions, since each token represents a separate observation and action.

In [12]:
def calc_kl_penalty(
    logits: Float[Tensor, "minibatch_size gen_len d_vocab"],
    ref_logits: Float[Tensor, "minibatch_size gen_len d_vocab"],
    kl_coef: float,
    gen_len: int,
) -> Float[Tensor, ""]:
    """
    Computes the KL divergence between the logits and the reference logits, scaled
    by the penalty function. This is used to stop the learned policy from diverging
    too much from the original reference model's policy.

    Args:
        logits:
            The logits for all generated tokens (under the new model).
        ref_logits:
            The logits for the generated tokens (under the reference model).
        kl_coef:
            The coefficient of the KL penalty.
        gen_len:
            the number of generated tokens (i.e. the number of tokens we want to compute kl penalty for)

    Output:
        The KL divergence between the logits and the reference logits, scaled by kl_coef.
    """
    assert logits.shape[1] == ref_logits.shape[1] == gen_len, (
        "Should pass in logits & ref_logits for generated tokens only, i.e. [:, -gen_len-1: -1]"
    )
    logprobs = t.log_softmax(logits, dim=-1) # batch, gen_len, d_vocab
    probs = logprobs.exp() # batch, gen_len, d_vocab
    ref_logprobs = t.log_softmax(ref_logits, dim=-1) # batch, gen_len, d_vocab
    kl_div = (probs * (logprobs - ref_logprobs)) # batch, gen_len, d_vocab
    kl_div = kl_div.sum(dim=-1) # batch, gen_len
    return kl_coef * kl_div.mean()


tests.test_calc_kl_penalty(calc_kl_penalty)
tests.test_calc_kl_penalty_stability(calc_kl_penalty)

All tests in `test_calc_kl_penalty` passed!
All tests in `test_calc_kl_penalty_stability` passed!


<details><summary>Solution</summary>

```python
def calc_kl_penalty(
    logits: Float[Tensor, "minibatch_size gen_len d_vocab"],
    ref_logits: Float[Tensor, "minibatch_size gen_len d_vocab"],
    kl_coef: float,
    gen_len: int,
) -> Float[Tensor, ""]:
    """
    Computes the KL divergence between the logits and the reference logits, scaled
    by the penalty function. This is used to stop the learned policy from diverging
    too much from the original reference model's policy.

    Args:
        logits:
            The logits for all generated tokens (under the new model).
        ref_logits:
            The logits for the generated tokens (under the reference model).
        kl_coef:
            The coefficient of the KL penalty.
        gen_len:
            the number of generated tokens (i.e. the number of tokens we want to compute kl penalty for)

    Output:
        The KL divergence between the logits and the reference logits, scaled by kl_coef.
    """
    assert logits.shape[1] == ref_logits.shape[1] == gen_len, (
        "Should pass in logits & ref_logits for generated tokens only, i.e. [:, -gen_len-1: -1]"
    )

    ref_logprobs = ref_logits.log_softmax(-1)
    logprobs = logits.log_softmax(-1)
    probs = logprobs.exp()

    kl_div = (probs * (logprobs - ref_logprobs)).sum(-1)

    return kl_coef * kl_div.mean()
```
</details>

### Exercise - (re)implement `compute_entropy_bonus`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

Next, we'll implement the entropy bonus function again. Rather than working with `probs.entropy()` like yesterday, we'll need to compute entropy directly from the logits, and **take the mean over batch and sequence position dimensions**.

The formula for entropy of a distribution $P$ is $- \sum_i P_i \log P_i$. You'll need to take the same numerical stability precautions as the previous exercise.

In [13]:
def calc_entropy_bonus(
    logits: Float[Tensor, "minibatch_size gen_len d_vocab"], ent_coef: float, gen_len: int
) -> Float[Tensor, ""]:
    """
    Return the entropy bonus term, suitable for gradient ascent.

    Args:
        logits:
            the logits of the tokens generated by the model before each generated token
        ent_coef:
            the coefficient for the entropy loss, which weights its contribution to the overall
            objective function.
        gen_len:
            the number of generated tokens (i.e. the number of tokens we want to compute the entropy
            bonus for).
    """
    assert logits.shape[1] == gen_len, "Should pass in logits *before* all generated tokens, i.e. [:, -gen_len-1: -1]"
    logprobs = t.log_softmax(logits, dim=-1)
    probs = logprobs.exp()
    ent = -(probs * logprobs).sum(-1).mean()
    return ent_coef * ent


tests.test_calc_entropy_bonus(calc_entropy_bonus)
tests.test_calc_entropy_bonus_stability(calc_entropy_bonus)

All tests in `test_calc_entropy_bonus` passed!
All tests in `test_calc_entropy_bonus_stability` passed!


<details><summary>Solution</summary>

```python
def calc_entropy_bonus(
    logits: Float[Tensor, "minibatch_size gen_len d_vocab"], ent_coef: float, gen_len: int
) -> Float[Tensor, ""]:
    """
    Return the entropy bonus term, suitable for gradient ascent.

    Args:
        logits:
            the logits of the tokens generated by the model before each generated token
        ent_coef:
            the coefficient for the entropy loss, which weights its contribution to the overall
            objective function.
        gen_len:
            the number of generated tokens (i.e. the number of tokens we want to compute the entropy
            bonus for).
    """
    assert logits.shape[1] == gen_len, "Should pass in logits *before* all generated tokens, i.e. [:, -gen_len-1: -1]"

    logprobs = logits.log_softmax(dim=-1)
    probs = logprobs.exp()
    entropy = -(probs * logprobs).sum(dim=-1)
    return ent_coef * entropy.mean()
```
</details>

### Other objective function terms

Since the other two terms in our objective function (value function loss and clipped surrogate objective) are pretty much identical to yesterday's, we've provided them for you (taken from yesterday's solutions code). We've added some extra comments in the docstrings to highlight how they differ from yesterday's PPO implementation.

You should pay attention to the shapes of the inputs to these functions (in particular whether they're shape `seq_len` meaning they're for all tokens, or `gen_len` meaning they're only for tokens after the prefix), so that you use them correctly when you're writing the `RLHFTrainer` methods.

In [14]:
def calc_value_function_loss(
    values: Float[Tensor, "minibatch_size gen_len"],
    mb_returns: Float[Tensor, "minibatch_size gen_len"],
    vf_coef: float,
    gen_len: int,
) -> Float[Tensor, ""]:
    """Compute the value function portion of the loss function.

    Note that for RLHF with advantages = TD residuals rather than GAE, this is equivalent to
    penalizing the squared error between values[t] and mb_values[t+1]. This is essentially
    equivalent to our TD loss expression for DQN, where we penalized the current network's Q values
    and the next-step target network Q values. The role is the same in both cases: to improve the
    accuracy (and reduce the variance) of our value function estimates.

    values:
        the value function predictions for the sampled minibatch, for all generated tokens (using
        the updated critic network).
    mb_returns:
        the target for our updated critic network (computed as `advantages + values` from the old
        network).
    vf_coef:
        the coefficient for the value loss, which weights its contribution to the overall loss.
        Denoted by c_1 in the paper.
    gen_len:
        the number of generated tokens, used for shape checking
    """
    assert values.shape[1] == gen_len, "Should pass in values before all generated tokens, i.e. [:, -gen_len-1: -1]"
    assert mb_returns.shape[1] == gen_len, "Should pass in returns before all generated tokens only"

    return 0.5 * vf_coef * (values - mb_returns).pow(2).mean()


def calc_clipped_surrogate_objective(
    logprobs: Float[Tensor, "minibatch_size gen_len"],
    mb_logprobs: Float[Tensor, "minibatch_size gen_len"],
    mb_advantages: Float[Tensor, "minibatch_size gen_len"],
    clip_coef: float,
    gen_len: int,
    eps: float = 1e-8,
) -> Float[Tensor, ""]:
    """Return the clipped surrogate objective, suitable for maximisation with gradient ascent.

    Note that for RLHF, we only care about the logprobs for the generated tokens, i.e. after the
    prefix. This is because we're fixing the prefix tokens and the model can't change its output for
    them, so there's no point including these in our objective function.

    logprobs:
        the logprobs of the action taken by the agent, according to the new policy
    mb_logprobs:
        logprobs of the actions taken in the sampled minibatch (according to the old policy)
    mb_advantages:
        advantages calculated from the sampled minibatch
    clip_coef:
        amount of clipping, denoted by epsilon in Eq 7.
    gen_len:
        the number of generated tokens, used for shape checking
    eps:
        used to add to std dev of mb_advantages when normalizing (to avoid dividing by zero)
    """
    assert logprobs.shape[1] == mb_logprobs.shape[1] == mb_advantages.shape[1] == gen_len, (
        "Should pass in logprob/advantage data for generated tokens only, i.e. [:, -gen_len-1: -1]"
    )

    logits_diff = logprobs - mb_logprobs

    r_theta = t.exp(logits_diff)

    mb_advantages = normalize_reward(mb_advantages, eps)

    non_clipped = r_theta * mb_advantages
    clipped = t.clip(r_theta, 1 - clip_coef, 1 + clip_coef) * mb_advantages

    return t.minimum(non_clipped, clipped).mean()

### Exercise - implement `get_logprobs`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

You'll notice that the functions above take logprobs of shape `(minibatch_size, gen_len)`, i.e. the logprobs on correct tokens for all the tokens generated by the model. This is because we don't care about the logprobs the model assigns to the prefix tokens, since it's not in control of them. So you'll find it useful to implement the function `get_logprobs` below, which returns the logprobs for the correct tokens _after_ the prefix. For example:

- If `prefix_len = 1` then all the model's logprobs are predicting non-prefix tokens, so we return `logprobs[:, :-1]` indexed at the non-prefix correct next tokens i.e. `tokens[:, 1:]`. The return type has shape `(batch, seq_len-1)`.
- If `prefix_len = 2` then we discard the very first logprob because it's predicting part of the prefix not new actions, so we return `logprobs[:, 1:-1]` indexed at the non-prefix correct next tokens i.e. `tokens[:, 2:]`. The return type has shape `(batch, seq_len-2)`.

When `prefix_len` is `None` you should have the same behaviour as if `prefix_len = 1`, i.e. returning `seq_len-1` correct logprobs.

<!-- <img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/get-correct-logprobs-3-solid.png" width="520"> -->

You can implement this function using regular indexing, tools like `torch.gather`, or with the `eindex` library which should be included in your dependencies (see [here](https://www.perfectlynormal.co.uk/blog-eindex) for how to use this library).

In [15]:
def get_logprobs(
    logits: Float[Tensor, "batch seq_len vocab"],
    tokens: Int[Tensor, "batch seq_len"],
    prefix_len: int | None = None,
) -> Float[Tensor, "batch gen_len"]:
    """
    Returns correct logprobs for the given logits and tokens, for all the tokens after the prefix
    tokens (which have length equal to `prefix_len`).

    If prefix_len = None then we return shape (batch, seq_len-1).
    If not, then we return shape (batch, seq_len-prefix_len) representing the predictions for all
    toks after the prefix.
    """
    # slice to only gen_len
    if prefix_len is not None:
        logits = logits[:, prefix_len - 1:, :]
        tokens = tokens[:, prefix_len - 1:]
    logprobs = t.log_softmax(logits, dim=-1)
    # index logprobs by the next token
    # action_logprobs[t] = logprobs[tokens[t+1]]
    return eindex(logprobs, tokens, 'batch seq_len [batch seq_len+1]')


tests.test_get_logprobs(get_logprobs)

All tests for `get_logprobs` passed (for prefix_len = None)!
All tests for `get_logprobs` passed (for prefix_len > 0)!


/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:288: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  assert torch.tensor(shape).prod().item() == index_tensor[idx].numel(), \
/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:292: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexi

<details><summary>Solution</summary>

```python
def get_logprobs(
    logits: Float[Tensor, "batch seq_len vocab"],
    tokens: Int[Tensor, "batch seq_len"],
    prefix_len: int | None = None,
) -> Float[Tensor, "batch gen_len"]:
    """
    Returns correct logprobs for the given logits and tokens, for all the tokens after the prefix
    tokens (which have length equal to `prefix_len`).

    If prefix_len = None then we return shape (batch, seq_len-1).
    If not, then we return shape (batch, seq_len-prefix_len) representing the predictions for all
    toks after the prefix.
    """
    # Slice our tensors based on prefix_len
    if prefix_len is not None:
        logits = logits[:, prefix_len - 1 :]
        tokens = tokens[:, prefix_len - 1 :]

    # Get logprobs
    logprobs = logits.log_softmax(-1)

    # We want to get elements `logprobs[b, s, tokens[b, s+1]]`, we do this using eindex as follows:
    correct_logprobs = eindex(logprobs, tokens, "b s [b s+1]")

    return correct_logprobs
```
</details>

## Optimizer & Scheduler

### Exercise - implement `get_optimizer`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

We need to be a bit careful when defining our optimizer. It makes no sense to have the same learning rate for our original model as we do for our value head. The value head was randomly initialized and has no idea what it's doing, but our model is pretrained and so it already has weights which have been trained to effectively extract features from text.

The syntax for using parameter groups in an optimizer is as follows:

```python
parameter_groups = [
    {"params": [param1, param2, ...], "lr": lr1},
    {"params": [param3, param4, ...], "lr": lr2},
]
```

where `params` is a list (or iterable) of parameters, and `lr` is the learning rate for these parameters.

You should fill in the function `get_optimizer` below, so that the value head's parameters all have learning rate `args.head_learning_rate` and the base model's parameters all have learning rate `args.base_learning_rate`.

Remember that we're using `maximize=True` with our optimizer (since we're maximizing an objective function rather than minimizing a loss function). Also we're using the `AdamW` optimizer (our implementation doesn't include weight decay so we could in theory use `Adam`, but it's better to stick to AdamW just in case we want to add in weight decay later).

In [16]:
def get_optimizer(model: HookedTransformerWithValueHead, base_lr: float, head_lr: float) -> t.optim.Optimizer:
    """
    Returns an AdamW optimizer for the model, with the correct learning rates for the base and head.
    Make sure to use the HookedTransformerWithValueHead wrapper methods for getting the parameters.
    """
    return t.optim.AdamW([
        {
            "params": model.get_base_model_trainable_params(),
            "lr": base_lr
        },
        {
            "params": model.get_value_head_params(),
            "lr": head_lr
        }
    ], maximize=True)


tests.test_get_optimizer(get_optimizer, model)

All tests for `get_optimizer` passed!


<details><summary>Solution</summary>

```python
def get_optimizer(model: HookedTransformerWithValueHead, base_lr: float, head_lr: float) -> t.optim.Optimizer:
    """
    Returns an AdamW optimizer for the model, with the correct learning rates for the base and head.
    Make sure to use the HookedTransformerWithValueHead wrapper methods for getting the parameters.
    """
    return t.optim.AdamW(
        [
            {"params": model.get_base_model_trainable_params(), "lr": base_lr},
            {"params": model.get_value_head_params(), "lr": head_lr},
        ],
        maximize=True,
    )
```
</details>

### Scheduler

In PPO, we had you write a custom class for implementing learning rate scheduling. This was useful to help you engage with the low-level syntax of changing learning rates in Pytorch. However, PyTorch does provide a handy class for implementing custom learning rate scheduling:

```python
optimizer = t.optim.Adam(...)
scheduler = t.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
```

where `lr_lambda` is a function mapping the number of steps (i.e. number of times we've called `scheduler.step()`) to a float which **gets multiplied by the base learning rate** (i.e. 0.1 means we use 10% of the base LR). There are schedulers other than `LambdaLR` which have specific built-in behaviour (see [documentation page](https://pytorch.org/docs/stable/optim.html)), although this gives you the most flexibility.

<details>
<summary>Aside - why we use warmup</summary>

Warmup is a common strategy early in training, to make sure we don't get excessive updates early on. It seems to work pretty well empirically. Some possible reasons for this are:

* It helps avoid large updates when the Adam moving averages of first and second moments are not yet well calibrated.
* Early on in training, the gradients might be very large (especially for the value function) because the model's prediction is nowhere near where it needs to be. So an LR warmup is more useful early on, to help avoid massive steps.

</details>

We've given you the code you'll be using for returning a custom `lr_lambda` function with a **linear warmup then linear decay**. We've also provided code for you in the trainer class's init method below which creates your scheduler. All you need to do is make sure you're stepping it appropriately.

In [17]:
def get_optimizer_and_scheduler(args: RLHFArgs, model: HookedTransformerWithValueHead):
    """
    Creates an AdamW optimizer and an LR scheduler that linearly warms up for `warmup_steps` steps,
    and then linearly decays to `final_scale` over the remaining steps.
    """

    def lr_lambda(step):
        assert step <= args.total_phases, f"Step = {step} should be less than total_phases = {args.total_phases}."
        if step < args.warmup_steps:
            return step / args.warmup_steps
        else:
            return 1 - (1 - args.final_scale) * (step - args.warmup_steps) / (args.total_phases - args.warmup_steps)

    optimizer = get_optimizer(model, args.base_lr, args.head_lr)
    scheduler = t.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler

If we want to log the learning rate, then we can use `scheduler.get_last_lr()` which gives you a list of learning rates for each parameter group (in our case, this would have length 2).

## Training your model

We're now ready to put everything together! We've provided you with the template of a training loop which should be very similar to yesterday's.

### Exercise - complete `RLHFTrainer`

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 40-60 minutes on this exercise.
> ```

The `compute_rlhf_objective` method should be very similar to yesterday's `compute_ppo_objective` method (i.e. it should compute the 3 terms in the PPO objective function and combine them into a single objective function which gets returned), although there are a few small differences:

- You also need to compute the KL penalty term with `calc_kl_penalty` and include it in the objective function - make sure you get the correct sign!
- Rather than getting `logits` and `values` from your actor and critic models, you get them both from the `forward` method of your `TransformerWithValueHead` model. 
    - Also, make sure you pass in the correct slices to your `calc_...` objective functions (although they should flag if you've done this incorrectly via the assert statements at the start of these functions)

The `learning_phase` method should be identical to yesterday's `learning_phase` method (i.e. it should generate minibatches via `memory.get_minibatches()` and then iterate through them, performing a step of gradient ascent on each). The only thing you need to adjust is the scheduler step - the way we've set it up, this should be done once per phase, not once per step (this is generally more common practice in ML; we step with the scheduler once per epoch).

A few tips / notes before you start:

- For faster feedback loops, don't use `wandb` until you've stopped getting errors!
- You can log text to Weights & Biases: just printing normal output should appear under the "Logs" section, but if you want to see it with the rest of your wandb charts then you can also use [`wandb.Table`](https://docs.wandb.ai/guides/track/log/log-tables/) to log tables.

<!-- #### Logging text to wandb

If you want to log text to Weights & Biases, there are 2 main ways:

1. Just print output, this is logged to weights & biases under the "Logs" section!
2. Log tables. This should usually be done just once at the end of training (because you can't log tables incrementally, only all at once). Here's some example code I used here for logging all my samples in a single table, as well as my hyperparameters (useful when creating a run report):

```python
wandb.log({
    "samples_table": wandb.Table(["sample"], self.samples),
    "config_params": wandb.Table(["param", "values"], [[k, v.__name__ if callable(v) else str(v)] for k, v in self.args.__dict__.items()])
})
```

This works when `self.samples` is a list of length-1 lists, each containing a single sample (i.e. one of the strings returned frmo the `get_samples` method). -->

In [20]:
class RLHFTrainer:
    model: HookedTransformerWithValueHead
    ref_model: HookedTransformer
    memory: ReplayMemory  # we'll set this during rollout

    def __init__(self, args: RLHFArgs):
        t.manual_seed(args.seed)
        self.args = args
        self.run_name = f"{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        self.model = HookedTransformerWithValueHead.from_pretrained(args.base_model).to(device).train()
        self.ref_model = HookedTransformer.from_pretrained(args.base_model).to(device).eval()
        self.optimizer, self.scheduler = get_optimizer_and_scheduler(self.args, self.model)
        self.prefix_len = len(self.model.to_str_tokens(self.args.prefix, prepend_bos=self.args.prepend_bos))

    def compute_rlhf_objective(self, minibatch: ReplayMinibatch):
        """
        Computes the RLHF objective function to maximize, which equals the PPO objective function
        modified by the KL penalty term.

        Steps of this function are:
            - Get logits & values for the samples in minibatch
            - Get the logprobs of the minibatch actions taken
            - Use this data to compute all 4 terms of the RLHF objective function, and return it
            - Also optionally log stuff to Weights & Biases (and print some sample completions)
        """
        gen_len_slice = slice(-self.args.gen_len - 1, -1)
        logits, values = self.model.forward_with_value_head(minibatch.sample_ids)
        gen_logprobs = get_logprobs(logits, minibatch.sample_ids, self.prefix_len)
        clipped_surrogate_obj = calc_clipped_surrogate_objective(
            gen_logprobs,
            minibatch.logprobs,
            minibatch.advantages,
            self.args.clip_coef,
            self.args.gen_len
        )
        value_loss = calc_value_function_loss(
            values[:, gen_len_slice], 
            minibatch.returns, 
            self.args.vf_coef, 
            self.args.gen_len
        )
        ent_bonus = calc_entropy_bonus(
            logits[:, gen_len_slice],
            self.args.ent_coef, 
            self.args.gen_len
        )
        kl_penalty = calc_kl_penalty(
            logits[:, gen_len_slice],
            minibatch.ref_logits[:, gen_len_slice],
            self.args.kl_coef,
            self.args.gen_len
        )

        with t.inference_mode():
            ratio = (gen_logprobs - minibatch.logprobs).exp()
            clipfrac = ((ratio - 1.0).abs() > self.args.clip_coef).float().mean().item()
        if self.args.use_wandb:
            wandb.log(
                dict(
                    total_steps=self.step,
                    lr=self.scheduler.get_last_lr()[0],
                    clipped_surrogate_objective=clipped_surrogate_obj.item(),
                    clipfrac=clipfrac,
                    value_loss=value_loss.item(),
                    values=values.mean().item(),
                    entropy_bonus=ent_bonus.item(),
                    kl_penalty=kl_penalty.item()
                ),
                step=self.step
            )

        return clipped_surrogate_obj + ent_bonus - value_loss - kl_penalty

    def rollout_phase(self) -> ReplayMemory:
        """
        Performs a single rollout phase, returning a ReplayMemory object containing the data
        generated during this phase. Note that all forward passes here should be done in inference
        mode.

        Steps of this function are:
            - Generate samples from our model
            - Get logits of those generated samples (from model & reference model)
            - Get other data for memory (logprobs, normalized rewards, advantages)
            - Return this data in a ReplayMemory object
        """
        with t.inference_mode():
            sample_ids, samples = get_samples(
                self.model,
                prompt=self.args.prefix,
                batch_size=self.args.batch_size,
                gen_len=self.args.gen_len,
                temperature=self.args.temperature,
                top_k=self.args.top_k,
                prepend_bos=self.args.prepend_bos,
                verbose=False,
            )
            logits, values = self.model.forward_with_value_head(sample_ids)
            ref_logits = self.ref_model(sample_ids)
        rewards = self.args.reward_fn(samples)
        rewards_normed = normalize_reward(rewards) if self.args.normalize_reward else rewards
        logprobs = get_logprobs(logits, sample_ids, self.prefix_len)
        advantages = compute_advantages(values, rewards_normed, self.prefix_len)

        # log stuff
        rewards_mean = rewards.mean()
        if self.args.use_wandb:
            wandb.log({"mean_reward": rewards_mean.item()}, step=self.step)
        n_log_samples = min(3, self.args.batch_size)
        ref_logprobs = get_logprobs(ref_logits[:n_log_samples], sample_ids[:n_log_samples], self.prefix_len).sum(-1)
        headers = ["Reward", "Ref logprobs", "Sample"]
        table_data = [[str(int(r)), f"{lp:.2f}", repr(s)] for r, lp, s in zip(rewards.tolist(), ref_logprobs, samples)]
        table = tabulate(table_data, headers, tablefmt="simple_grid", maxcolwidths=[None, None, 90])
        print(f"Phase {self.phase+1:03}/{self.args.total_phases}, Mean reward: {rewards_mean:.4f}\n{table}\n")

        return ReplayMemory(self.args, sample_ids, logprobs, advantages, values, ref_logits)

    def learning_phase(self, memory: ReplayMemory) -> float:
        """
        Performs a learning step on `memory`. This involves the standard gradient descent steps
        (i.e. zeroing gradient, computing objective function, doing backprop, stepping optimizer).

        def train(self) -> None:
            - Clipping grad norm to the value given in `self.args.max_grad_norm`
            - Incrementing `self.step` by  iidwjwdlkwj;aefefeffeefewfasss
              each minibatch
            - Stepping the scheduler (once per calling of this function)

        Returns the average objective function value over the minibatches as a float for logging.
        """
        minibatches = memory.get_minibatches()
        total_obj = t.zeros((1,), dtype=t.float32, device=device)
        for minibatch in minibatches:
            obj = self.compute_rlhf_objective(minibatch)
            obj.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.args.max_grad_norm)
            total_obj += obj
            self.optimizer.step()
            self.optimizer.zero_grad()
            self.step += 1
        self.scheduler.step()
        return (total_obj / len(minibatches)).item()

    def train(self) -> None:
        """
        Performs a full training run.
        """
        self.step = 0
        self.samples = []

        if self.args.use_wandb:
            wandb.init(
                project=self.args.wandb_project_name,
                entity=self.args.wandb_entity,
                name=self.run_name,
                config=self.args,
            )
        runner = tqdm(range(self.args.total_phases))
        for self.phase in runner:
            memory = self.rollout_phase()
            loss = self.learning_phase(memory)
            runner.set_description(f"Loss: {loss:.4f}")

        if self.args.use_wandb:
            wandb.finish()

<details>
<summary>Solution (simpler, no logging)</summary>

```python
def compute_rlhf_objective(self, minibatch: ReplayMinibatch):
    gen_len_slice = slice(-self.args.gen_len - 1, -1)  # define this for convenience

    # Get logits & values for our generated minibatch samples
    logits, values = self.model(minibatch.sample_ids)

    # Get logprobs for the the tokens generated (i.e. the logprobs of our actions)
    logprobs = get_logprobs(logits, minibatch.sample_ids, self.prefix_len)

    # Compute all terms of the loss function (including KL penalty)
    clipped_surrogate_objective = calc_clipped_surrogate_objective(
        logprobs, minibatch.logprobs, minibatch.advantages, self.args.clip_coef, self.args.gen_len
    )
    value_loss = calc_value_function_loss(
        values[:, gen_len_slice], minibatch.returns, self.args.vf_coef, self.args.gen_len
    )
    entropy_bonus = calc_entropy_bonus(logits[:, gen_len_slice], self.args.ent_coef, self.args.gen_len)
    kl_penalty = calc_kl_penalty(
        logits[:, gen_len_slice], minibatch.ref_logits[:, gen_len_slice], self.args.kl_coef, self.args.gen_len
    )

    # Compute net objective function
    ppo_objective_fn = clipped_surrogate_objective - value_loss + entropy_bonus
    total_objective_function = ppo_objective_fn - kl_penalty

    return total_objective_function

def rollout_phase(self) -> ReplayMemory:
    # Get our samples
    sample_ids, samples = get_samples(
        self.model.base_model,
        prompt=self.args.prefix,
        batch_size=self.args.batch_size,
        gen_len=self.args.gen_len,
        temperature=self.args.temperature,
        top_k=self.args.top_k,
        prepend_bos=self.args.prepend_bos,
    )
    # Generate logits from our model & reference model
    with t.inference_mode():
        logits, values = self.model(sample_ids)
        ref_logits = self.ref_model(sample_ids)

    # Get the logprobs of the generated tokens
    logprobs = get_logprobs(logits, sample_ids, self.prefix_len)

    # Calculate & normalize rewards (note we don't normalize inplace, because we want to log unnormalized rewards)
    rewards = self.args.reward_fn(samples)
    rewards_mean = rewards.mean().item()
    rewards_normed = normalize_reward(rewards) if self.args.normalize_reward else rewards

    # Compute advantages
    advantages = compute_advantages(values, rewards_normed, self.prefix_len)

    return ReplayMemory(
        args=self.args,
        sample_ids=sample_ids,
        logprobs=logprobs,
        advantages=advantages,
        values=values,
        ref_logits=ref_logits,
    )

def learning_phase(self, memory: ReplayMemory) -> None:
    for minibatch in memory.get_minibatches():
        self.optimizer.zero_grad()
        total_objective_function = self.compute_rlhf_objective(minibatch)
        total_objective_function.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.args.max_grad_norm)
        self.optimizer.step()
        self.step += 1

    self.scheduler.step()
```

</details>

<details>
<summary>Solution (full, with logging)</summary>

```python
def compute_rlhf_objective(self, minibatch: ReplayMinibatch):
    gen_len_slice = slice(-self.args.gen_len - 1, -1)  # define this for convenience

    # Get logits & values for our generated minibatch samples
    logits, values = self.model(minibatch.sample_ids)

    # Get logprobs for the the tokens generated (i.e. the logprobs of our actions)
    logprobs = get_logprobs(logits, minibatch.sample_ids, self.prefix_len)

    # Compute all terms of the loss function (including KL penalty)
    clipped_surrogate_objective = calc_clipped_surrogate_objective(
        logprobs, minibatch.logprobs, minibatch.advantages, self.args.clip_coef, self.args.gen_len
    )
    value_loss = calc_value_function_loss(
        values[:, gen_len_slice], minibatch.returns, self.args.vf_coef, self.args.gen_len
    )
    entropy_bonus = calc_entropy_bonus(logits[:, gen_len_slice], self.args.ent_coef, self.args.gen_len)
    kl_penalty = calc_kl_penalty(
        logits[:, gen_len_slice], minibatch.ref_logits[:, gen_len_slice], self.args.kl_coef, self.args.gen_len
    )

    # Compute net objective function
    ppo_objective_fn = clipped_surrogate_objective - value_loss + entropy_bonus
    total_objective_function = ppo_objective_fn - kl_penalty

    # Log stuff
    with t.inference_mode():
        logratio = logprobs - minibatch.logprobs
        ratio = logratio.exp()
        clipfracs = [((ratio - 1.0).abs() > self.args.clip_coef).float().mean().item()]
    if self.args.use_wandb:
        wandb.log(
            dict(
                total_steps=self.step,
                lr=self.scheduler.get_last_lr()[0],
                clipped_surrogate_objective=clipped_surrogate_objective.item(),
                clipfrac=np.mean(clipfracs),
                value_loss=value_loss.item(),
                values=values.mean().item(),
                entropy_bonus=entropy_bonus.item(),
                kl_penalty=kl_penalty.item(),
            ),
            step=self.step,
        )

    return total_objective_function

def rollout_phase(self) -> ReplayMemory:
    # Get our samples
    sample_ids, samples = get_samples(
        self.model.base_model,
        prompt=self.args.prefix,
        batch_size=self.args.batch_size,
        gen_len=self.args.gen_len,
        temperature=self.args.temperature,
        top_k=self.args.top_k,
        prepend_bos=self.args.prepend_bos,
    )
    # Generate logits from our model & reference model
    with t.inference_mode():
        logits, values = self.model(sample_ids)
        ref_logits = self.ref_model(sample_ids)

    # Get the logprobs of the generated tokens
    logprobs = get_logprobs(logits, sample_ids, self.prefix_len)

    # Calculate & normalize rewards (note we don't normalize inplace, because we want to log unnormalized rewards)
    rewards = self.args.reward_fn(samples)
    rewards_mean = rewards.mean().item()
    rewards_normed = normalize_reward(rewards) if self.args.normalize_reward else rewards

    # Compute advantages
    advantages = compute_advantages(values, rewards_normed, self.prefix_len)

    # Log stuff, and print output in a readable way (you could easily just regular print here instead of rprint table)
    if self.args.use_wandb:
        wandb.log({"mean_reward": rewards_mean}, step=self.step)

    n_log_samples = min(3, self.args.batch_size)
    ref_logprobs = get_logprobs(ref_logits[:n_log_samples], sample_ids[:n_log_samples], self.prefix_len).sum(-1)
    headers = ["Reward", "Ref logprobs", "Sample"]
    table_data = [[str(int(r)), f"{lp:.2f}", repr(s)] for r, lp, s in zip(rewards.tolist(), ref_logprobs, samples)]
    table = tabulate(table_data, headers, tablefmt="simple_grid", maxcolwidths=[None, None, 90])
    print(f"Phase {self.phase+1:03}/{self.args.total_phases}, Mean reward: {rewards_mean:.4f}\n{table}\n")

    return ReplayMemory(
        args=self.args,
        sample_ids=sample_ids,
        logprobs=logprobs,
        advantages=advantages,
        values=values,
        ref_logits=ref_logits,
    )

def learning_phase(self, memory: ReplayMemory) -> None:
    for minibatch in memory.get_minibatches():
        self.optimizer.zero_grad()
        total_objective_function = self.compute_rlhf_objective(minibatch)
        total_objective_function.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.args.max_grad_norm)
        self.optimizer.step()
        self.step += 1

    self.scheduler.step()
```

</details>

Once you've implemented your trainer class, you can run the code below to train your model. We recommend you start with the test run below, using a KL coefficient of zero.

<details>
<summary>Question - with <code>kl_coef=0.0</code>, what results do you think you should reliably get?</summary>

With this KL coefficient, the model has no incentive to match the reference distribution, it will only try to maximize the reward. So once it's figured out that it can just output full stops all the time and totally abandon any kind of grammar or coherence, it will do this. By the end of 30 phases, the model should have collapsed into producing reward-maximizing output like `"This is......"`, or something close.

</details>

In [ ]:
# Testing your setup: kl_coef=0.0 (see dropdown above the previous code block for explanation)
if RUN_BASE_RLHF:
    args = RLHFArgs(use_wandb=False, kl_coef=0.0, total_phases=30, warmup_steps=0, reward_fn=reward_fn_char_count)
    trainer = RLHFTrainer(args)
    trainer.train()
else:
    print(f"{RUN_BASE_RLHF=}, skipping test run")

Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda
Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda


  0%|          | 0/30 [00:00<?, ?it/s]/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:288: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  assert torch.tensor(shape).prod().item() == index_tensor[idx].numel(), \
/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:292: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torc

Phase 001/30, Mean reward: 1.3047
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │         -71.8  │ '<|endoftext|>This is a very long post, as I want to talk about the differences between    │
│          │                │ different languages. This is my first post on these things, and it was very'               │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │         -62.77 │ '<|endoftext|>This is a list of the characters who appear in the series. For other         │
│          │                │ characters, check out their Wikipedia page.\n\nThe first episode (the first

Loss: 0.0122:   3%|▎         | 1/30 [00:02<01:25,  2.95s/it]

Phase 002/30, Mean reward: 1.2656
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │         -73.53 │ "<|endoftext|>This is not just a matter of a single incident. It's a national problem. The │
│          │                │ government is trying to keep up appearances that it is trying to deal with"                │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │         -61.74 │ "<|endoftext|>This is the first in a new series of articles by the University of Texas at  │
│          │                │ Austin's David A. Smith. Smith, a doctoral candidate in psychology, is"    

...

Loss: 0.0195:  93%|█████████▎| 28/30 [01:22<00:05,  2.99s/it]

Phase 029/30, Mean reward: 7.5625
┌──────────┬────────────────┬───────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                │
├──────────┼────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│        6 │         -92.49 │ '<|endoftext|>This is not.NotNotNotNot.NotNotNotNotNotNotNotNotNotNotNot.NotNotnotNot │
│          │                │ Not.How. . Not Not'                                                                   │
├──────────┼────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│        8 │         -87.02 │ '<|endoftext|>This is not.NotNotNot.NotNotNotNotNotNot.NotNotNot.notNot Not Not.      │
│          │                │ Not.NotNot. Not Not.'                                                                 │
├──────────┼──────────

Loss: 0.0208:  97%|█████████▋| 29/30 [01:25<00:02,  2.97s/it]

Phase 030/30, Mean reward: 9.2344
┌──────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                  │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │        -136.79 │ '<|endoftext|>This is not. Not..NotNotNotNotNotNotNotNotNot.NotNot.                     │
│          │                │ NotNotNotNot...<|endoftext|>It Not NotNot'                                              │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │         -98.13 │ '<|endoftext|>This is not.NotNotNotNotNot.NotNot.Not.NotNotNotNot.Not..NotNotNot.notNot │
│          │                │ Not..'                                                                                  │
├─────

Loss: 0.0182: 100%|██████████| 30/30 [01:28<00:00,  2.96s/it]


Once you've got this working, you can move on to a "proper run".

In [23]:
if RUN_BASE_RLHF:
    args = RLHFArgs(use_wandb=True, reward_fn=reward_fn_char_count, prefix="Help ")  # CUDA errors? reduce batch_size or gen_len
    trainer = RLHFTrainer(args)
    trainer.train()
else:
    print(f"{RUN_BASE_RLHF=}, skipping test run")

Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda
Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda


  0%|          | 0/100 [00:00<?, ?it/s]/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:288: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  assert torch.tensor(shape).prod().item() == index_tensor[idx].numel(), \
/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:292: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/tor

Phase 001/100, Mean reward: 0.8125
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │         -69.52 │ "<|endoftext|>Help !!!! I have a problem with my camera that is causing my phone to freeze │
│          │                │ or get lost in my bag. I don't want to pay more then"                                      │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │         -67.9  │ "<|endoftext|>Help ive been waiting for so long\n\ni've been waiting for a good quality    │
│          │                │ keyboard since I purchased my first one\n\nits so comfortable to type"    

Loss: 0.0000:   1%|          | 1/100 [00:03<04:58,  3.01s/it]

Phase 002/100, Mean reward: 0.7344
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │         -58.95 │ "<|endoftext|>Help !!!\n\nI have a problem with an image. The problem is with my computer, │
│          │                │ and I'm looking for a solution.\n\nI have"                                                 │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │         -31.87 │ '<|endoftext|>Help \ue005 \ue001 \ue007 \ue008 \ue009 \ue001 \ue002 \ue003 \ue004 \ue006 ' │
├──────────┼────────────────┼───────────────────────────────────────────────────────────────────────────

...

Phase 099/100, Mean reward: 1.8438
┌──────────┬────────────────┬──────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                   │
├──────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │         -61.14 │ "<|endoftext|>Help !!!\n\nIf you're having troubles getting this plugin to work, please  │
│          │                │ visit my other plugin page, http://tampermonkey.github."                                 │
├──────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │         -51.8  │ "<|endoftext|>Help !!!\n\n\nPlease don't ask me for help.\n\n\nI don't know what you've  │
│          │                │ done or where you came from, but I know"                                                

Loss: -0.0266:  99%|█████████▉| 99/100 [04:54<00:02,  2.95s/it]

Phase 100/100, Mean reward: 5.1797
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │         -63.14 │ '<|endoftext|>Help ____________________________________________________\n\nThe new version │
│          │                │ of the app is in the works. It will be much better and faster.\n\nPlease send any          │
│          │                │ feedback'                                                                                  │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │         -67.76 │ '<|endoftext|>Help !!!\nHere is an article on how to do a simple and easy 

Loss: -0.0315: 100%|██████████| 100/100 [04:56<00:00,  2.97s/it]


clipfrac,▂▁▁▄▁▁█▁▅▁▅▁▂▄▁▁▂▁▃▃▁▁▂▃▁▁▁▃▁▁▂▁▂▁▁▁▁▁▁▁
clipped_surrogate_objective,▁▂▁▄▂▅▂▅▂▂▃▂▃▅▁▂▂▆▆▁▂▃▅█▂▂▄▁▆▅▁▆▄▂▁▅▄▄▂▂
entropy_bonus,▇▆▆▅▄▁▆▇▁▄█▆█▃▇▇▇▅▇▃▅▄▅▆▅▆▇▄▄▂▃▆▆▆▄▂▄▂▅▅
kl_penalty,▁▂▃▂▃▄▄▅▆▇▆▆█▆▇▆▇▆▆▅▆▆▆▅▅▆▆▆▇▅▆▆▇▅▅▅▅▄▄▄
lr,▁▁▁▃▅▆▇██▇▇▇▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂
mean_reward,▁▁▁▃▁▁▁▁▁▁▂▂▁▁▂▁▄▁▁▂▁▁▂▂▂▂▂▂▂▂▃▅█▄▃▄▇▇▄▂
total_steps,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
value_loss,▃▃▄▄▃▄▄▃▄▅█▄▅▅▃▃▅▄▄▃▃▂▃▄▆▂▄▃▃▃▃█▃▁▄▁▂█▆▁
values,▃▄▄▄▅▆█▅▆▅▄▅▃▃▁▂▂▂▃▃▂▃▃▂▂▃▃▃▄▄▃▄▅▄▄▄▄▄▄▄
clipfrac,0.00833
clipped_surrogate_objective,0.00349


<details>
<summary>Some observations on the example run above</summary>

In this example, we see some strategies that the model has learned to maximize number of periods, such as:

- Short sentences written tersely, e.g. `This is a rush transcript. Copy may not be in its final form.`
- Acronyms like `a.k.a.`
- Websites, like `democracynow.org`

Another important observation in this particular run is that the model showed **mode collapse**, where it excessively optimizes for a narrow set of responses or strategies which have been shown to have high rewards. In this case, those examples are common sequences which occur frequently in the model's training data (which is why the reference logprobs are so high). The most obvious example here is `This is a rush transcript ...` (a common prefix for online news articles) followed by `AMY GOODMAN: This is Democracy Now!, democracynow.org` (which is how all articles on the progressive journalism website democracynow start).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/democracynow.png" width="540">

</details>

You can also play around with the parameters - in particular, try a few different prefix strings. The behaviour of the model (e.g. which kinds of techniques it converges onto for period maximization) or whether it easily mode collapses into insanity can be highly dependent on the prefix string!

Some common strategies you should observe include:

- Shorter sentences
- Repeating `U.S.` or `U.S.A.` (using the prefix prompt `"There is"`, this seems to be by far the most common strategy)
- Library versions e.g. `Python 2.7.12` or `the 2.6.0.2 release`
- Names with initials e.g. `C. S. Lewis` or titles e.g. `Dr.` and `PhD.`
- Abbreviations e.g. `Data-R.A.R. series` or `"L.A. Times"`
- Decimals in numbers e.g. `9.5cm x 7.5 cm`
- Triple periods e.g. `the man . . . the woman . . .`

You might also observe increasingly incoherent mode collapse if you train for too long and don't regularize with a high KL penalty. Here are a few that I got:

- `This is really helpful. The U.S. U.S. U.S. U.S.`
- `This is the A.A.G.A.R.M.A.R.M.A.R.M.A.R.M`
- `This is my mother. . . me. . . . . . . . . . . . . . . . . . . . . . . .`

### Exercise - use a more complex reward function

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 30-50 minutes on this exercise.
> ```

> Note: You will need a lot more VRAM to proceed with many following exercises. With `LOW_GPU_MEM = True` it's just barely possible to do this with 24GB VRAM, but in general we would recommend at least 40GB for some breathing room. Don't worry if you can't run them, these exercises are mostly for playing around with the reward model. You've already conceptually gained pretty much everything about RLHF if you've completed the above. We just now replace our toy reward model with something more complex.

We recommend you experiment with a few different reward functions, in particular some sentiment-based reward functions which are based on pretrained text classification models. For example, we might use one of the following:

- [`lvwerra/distilbert-imdb`](https://huggingface.co/lvwerra/distilbert-imdb), which was trained to classify IMDB film reviews as positive or negative.
- [`cardiffnlp/twitter-roberta-base-sentiment`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment), which is a model trained on tweets and finetuned for sentiment analysis (categories are positive, neutral and negative).
- [`distilbert-base-uncased-emotion`](bhadresh-savani/distilbert-base-uncased-emotion), which was finetuned on the [Emotion Dataset for Emotion Recognition Tasks](https://www.kaggle.com/datasets/parulpandey/emotion-dataset), i.e. it's trained to classify text according to emotional tone (classes are sadness, joy, love, anger, fear and surprise).

Note that for some of these, you should be using a prompt string which is appropriate for the reward function you're fine-tuning on, e.g. `"This movie was really"` for the IMDB model. Similarly, you might also want to change other parameters e.g. generation length. You can find a list of other models [here](https://huggingface.co/models?filter=text-classification). Lastly, note that it's fine to use probabilities rather than logits or logit diffs as your reward signal, since the reward normalization means that you'll still get a good signal even as the probabilities get close to 1.

<!-- For reference, you can see the parameters & results of a positive-sentiment IMDB run [here](https://api.wandb.ai/links/callum-mcdougall/3a1bl3y4), and a negative-sentiment run [here](https://api.wandb.ai/links/callum-mcdougall/misa79ct). The code to generate these two outputs respectively can be found below: -->

We've given you a template below, for creating a reward function from the IMDB sentiment classification model. Your job is to complete this function.

In [26]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

if RUN_BASE_RLHF:
    assert not LOW_GPU_MEM, "You will need more memory to use the imdb reward model."
    cls_model = AutoModelForSequenceClassification.from_pretrained("lvwerra/distilbert-imdb").half().to(device)
    cls_tokenizer = AutoTokenizer.from_pretrained("lvwerra/distilbert-imdb")
else:
    print(f"{RUN_BASE_RLHF=}, skipping imdb reward model")


@t.no_grad()
def reward_fn_sentiment_imdb(
    gen_sample: list[str], direction: Literal["pos", "neg"] = "neg"
) -> Float[Tensor, " batch"]:
    """
    Reward function based on sentiment classification probability from the lvwerra/distilbert-imdb
    model.

    Args:
        gen_sample (list[str]): The generated sample to evaluate.
        direction (str): The sentiment of the reward function, either "pos" or "neg".
    """
    assert direction in ["pos", "neg"], "direction should be either 'pos' or 'neg'"

    tokens = cls_tokenizer(gen_sample, return_tensors="pt", padding=True, truncation=True)["input_ids"].to(device)
    padding = cls_model.config.pad_token_id
    logits = cls_model(tokens, attention_mask=(tokens != padding).float()).logits
    return logits.softmax(dim=-1)[:, 1 if direction == 'pos' else 0].to(device)



if RUN_BASE_RLHF:
    # Some samples taken from the IMDB dataset used to finetune this model
    samples = [
        "Just finished watching this movie for maybe the 7th or 8th time, picked it up one night previously viewed at Blockbuster and absolutely loved it, I've shown it to 4 people so far and they have enjoyed it as well.",
        "This was the most original movie I've seen in years. If you like unique thrillers that are influenced by film noir, then this is just the right cure for all of those Hollywood summer blockbusters clogging the theaters these days.",
        "I can't believe that those praising this movie herein aren't thinking of some other film.",
        "This film seemed way too long even at only 75 minutes.",
        "Really, I can't believe that I spent $5 on this movie. I am a huge zombie fanatic and thought the movie might be really good. It had zombies in it right? Was I wrong!",
    ]
    classes = ["pos", "pos", "neg", "neg", "neg"]

    reward_fn = partial(reward_fn_sentiment_imdb, direction="pos")
    sentiment = reward_fn(samples).tolist()

    table = Table(
        "Sample",
        "Classification",
        "Sentiment",
        title="Demo of `reward_fn_sentiment_imdb`",
        show_lines=True,
    )
    for sample, cls, sent in zip(samples, classes, sentiment):
        table.add_row(repr(sample), cls, f"{sent:.4f}")
    rprint(table)

                                        Demo of `reward_fn_sentiment_imdb`                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Sample                                                                             ┃ Classification ┃ Sentiment ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ "Just finished watching this movie for maybe the 7th or 8th time, picked it up one │ pos            │ 0.9941    │
│ night previously viewed at Blockbuster and absolutely loved it, I've shown it to 4 │                │           │
│ people so far and they have enjoyed it as well."                                   │                │           │
├────────────────────────────────────────────────────────────────────────────────────┼────────────────┼───────────┤
│ "This was the most original movie I've seen in years. If you like unique thrillers │ pos            │ 0.9883    │
│ that are influenced by film noir, then this is just the right cure for all of      │                │           │
│ those Hollywood summer blockbusters clogging the theaters these days."             │                │           │
├────────────────────────────────────────────────────────────────────────────────────┼────────────────┼───────────┤
│ "I can't believe that those praising this movie herein aren't thinking of some     │ neg            │ 0.1632    │
│ other film."                                                                       │                │           │
├────────────────────────────────────────────────────────────────────────────────────┼────────────────┼───────────┤
│ 'This film seemed way too long even at only 75 minutes.'                           │ neg            │ 0.0067    │
├────────────────────────────────────────────────────────────────────────────────────┼────────────────┼───────────┤
│ "Really, I can't believe that I spent $5 on this movie. I am a huge zombie fanatic │ neg            │ 0.0240    │
│ and thought the movie might be really good. It had zombies in it right? Was I      │                │           │
│ wrong!"                                                                            │                │           │
└────────────────────────────────────────────────────────────────────────────────────┴────────────────┴───────────┘

<details><summary>Solution</summary>

```python
@t.no_grad()
def reward_fn_sentiment_imdb(
    gen_sample: list[str], direction: Literal["pos", "neg"] = "pos"
) -> Float[Tensor, " batch"]:
    """
    Reward function based on sentiment classification probability from the lvwerra/distilbert-imdb
    model.

    Args:
        gen_sample (list[str]): The generated sample to evaluate.
        direction (str): The sentiment of the reward function, either "pos" or "neg".
    """
    assert direction in ["pos", "neg"], "direction should be either 'pos' or 'neg'"

    tokens = cls_tokenizer(gen_sample, return_tensors="pt", padding=True, truncation=True)["input_ids"].to(device)
    logits = cls_model(tokens).logits
    positive_cls = logits.softmax(dim=-1)[:, 1 if (direction == "pos") else 0]
    return positive_cls.to(device)
```
</details>

Once you've got this working, you can try and perform an actual run on positive / negative sentiment. We recommend using approximately 200 phases for this, and to generate about 50 tokens per sequence so you can get a good sense of what the review looks like.

In [28]:
if RUN_BASE_RLHF:
    args = RLHFArgs(use_wandb=True, reward_fn=reward_fn_sentiment_imdb, prefix="This movie was really", total_phases=200, gen_len=50, kl_coef=0.67)  # CUDA errors? reduce batch_size or gen_len
    trainer = RLHFTrainer(args)
    trainer.train()
else:
    print(f"{RUN_BASE_RLHF=}, skipping test run")

Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda
Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda


  0%|          | 0/200 [00:00<?, ?it/s]/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:288: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  assert torch.tensor(shape).prod().item() == index_tensor[idx].numel(), \
/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:292: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/tor

Phase 001/200, Mean reward: 0.1783
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │        -110.35 │ '<|endoftext|>This movie was really fun to watch. The premise is simple - two guys are     │
│          │                │ stuck at a train station, in the middle of the night, and they decide that they need money │
│          │                │ to buy their tickets. The film follows the men from the very first moment they meet one'   │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │         -82.95 │ "<|endoftext|>This movie was really great.\n\nI've heard great things abou

Loss: 0.0013:   0%|          | 1/200 [00:04<15:23,  4.64s/it]

Phase 002/200, Mean reward: 0.1740
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │        -101.02 │ "<|endoftext|>This movie was really good. It was so funny. The ending is so sad. The       │
│          │                │ ending was really sweet. I think that's what we're going to be waiting for. I'm looking    │
│          │                │ forward to what comes next. It's so much fun watching this movie with"                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │         -99.32 │ "<|endoftext|>This movie was really good! This movie was really good!\n\nT

...

Phase 199/200, Mean reward: 0.9771
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │        -109.68 │ "<|endoftext|>This movie was really bad, and it was bad because people thought that they   │
│          │                │ were seeing an adaptation of a novel. It was a very bad movie that was a total waste of    │
│          │                │ time and money. The characters were weak and the pacing just wasn't that great. It's"      │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │        -111.84 │ '<|endoftext|>This movie was really bad, the actors were horrible, the sto

Loss: -0.0561: 100%|█████████▉| 199/200 [15:26<00:04,  4.65s/it]

Phase 200/200, Mean reward: 0.9868
┌──────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                                                     │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │         -91.24 │ "<|endoftext|>This movie was really bad. It wasn't great. I don't even have a bad thing to │
│          │                │ say about it. The acting was terrible, the acting on the film was terrible and the         │
│          │                │ direction was terrible. This movie is not a good movie. The characters, the"               │
├──────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │         -99.74 │ "<|endoftext|>This movie was really bad. In fact it was probably the worst

Loss: -0.0561: 100%|██████████| 200/200 [15:31<00:00,  4.66s/it]


clipfrac,▁▄▁▇▁▆▇▁▁▁▅▇▆█▁▇▂▆▁▁▁▁▆▅▅▅▃▆▁▂▂▁▁▁▁▁▁▁▁▁
clipped_surrogate_objective,▂▄▂▇▁▇▆▇▁█▅▆▇▂▃▁▁▇▄▂▁▁▁▃▂▅▄▇▄▁▃▂▃▁▄▂▁▃▄▃
entropy_bonus,▆▇▆█▅▄▄▁▂▂▂▄▆▄▃▃▃▃▅▂▃▅▄▆▃▂▃▃▅▅▂▇▅▅▄▃▅▃▄▁
kl_penalty,▁▁▂▂▂▃▅▆▆▆▇▇▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇▇▇▇▇▆▆▇▇▇▇
lr,▁▄▇█████▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▅▅▅▄▄▄▃▃▂▂▂▂▂▂▁▁▁
mean_reward,▁▁▂▂▂▂▄▄▆▅▆▇▆▇▇▇▇▇▇▇▇███████████████████
total_steps,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇███
value_loss,▃▄█▃▃▃▂▄▂▃▂▃▂▂▃▁▁▂▃▃▂▃▂▃▂▂▂▅▃▃▄▁▂▃▁▁▁▂▆▁
values,▂▂▃▃▃▃██▆▆▆▅▆▅▅▆▆▅▅▄▄▄▄▅▄▃▃▃▃▂▂▂▂▂▂▁▂▁▁▁
clipfrac,0.0025
clipped_surrogate_objective,0.01444


# 2️⃣ LoRA Fine-Tuning

> ##### Learning Objectives
>
> - Understand the mechanism behind Low-Rank Adaptors, and how they allow for fine-tuning with less resources.
> - Implement LoRA in a transformer model.
> - Fine-tune larger models that would otherwise take too much VRAM to be possible.

**Go to the RLHF Training Args class we defined at the start of the previous section and set `RUN_BASE_RLHF = False`. This will skip all the expensive training runs for Section 1, so you can easily rerun the file.**

## Low-Rank Adaptors (🚧 Under construction 🚧)

For the previous section, we required to keep two copies of the model in memory: $\pi_{ppo}$ to train, and $\pi_{base}$ as a reference. Now, if our models are already large enough that it's maxing out the VRAM, we obviously can't realistically keep two copies of the model in memory.

Moreover, what is found in practice is that often the changes to the model are very minor, in that the activations before and after fine-tuning tend to be only a low-rank transformation. So, why not setup the fine-tuning setup such that only a low-rank transformation to the weights can be learned?

[LoRA (Low-Rank Adaptator)](https://arxiv.org/abs/2106.09685) allows us to fine-tune only a small number of additional parameters and keep the rest of the parameters fixed. We see in the diagram below that LoRA:

- Keeps the original linear layer weights $W : (d_{in} \times d_{out})$ fixed
- Adds additional low-rank matricies $A : (d_{in} \times r)$ and $B :(r \times d_{out})$, such that the fine-tuned linear layer $\tilde{W}$ performs the operation $\tilde{W}(x) =W(x) + B(A(x))$.


<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/LoraDiagram.png" width="640|">


Once training is done, we note that since both operations are linear, the final output is equivalent to adding a low-rank matrix to $W$. Once training is complete, one can set $\tilde{W} = W + BA$, which "bakes" the adaptor into the model. This means the final-finetuned model is architectually identical to the original.

### Exercise - complete `Lora`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Now, you'll implement the `Lora` class. This class should implement the basic LoRA block, which is a low-rank linear layer (no bias) written as two seperate matricies, a **project-down** matrix $A$ and a **project-up** matrix $B$.

For simplicity, the `Lora` module actually handles `n_inst` instances of a low-rank linear layer, to make it easier to interface with multi-head attention later on.

You should
* Finish the `__init__` method to define the model parameters `A` and `B`, and initalize then:
    * `A` should be initialized with `kaiming_uniform_` with $a = \sqrt{5}$.
    * `B` should be initialized with zeros.
* Implement the `forward` method to compute the forward pass of the LoRA block `f(x) = (x @ A) @ B * lora_alpha / rank`.
    * The larger the rank, the more we scale down the effect of the LoRA block.
    * `lora_alpha` is a hyperparameter that controls the scale of the LoRA block, usually quite large (32).

<details>
<summary>Why no bias?</summary>

There is no matrix $\tilde{W}$ for which $\tilde{W}x = Wx + b$ unless $b = 0$. Matrix multiplication is not an affine transformation.

</details>

<details>
<summary>Why `(x @ A) @ B` over `x @ (A @ B)`? </summary>

The product `A @ B` is of shape `(n_inst, d_in, d_out)`, making it quite a large matrix, but it can be of rank only at most `r`,
so it is very wasteful to perfom the multiplication in this order.

</details>

In [30]:
from torch.nn.init import kaiming_uniform, kaiming_uniform_


class Lora(nn.Module):
    """
    Module that implements the basic LoRA block.
    - Input: tensor of shape (..., [inst], d_in) and returns a tensor of shape (..., inst, d_out).
    - Calculated intermediate activations of shape (..., inst, rank)
    - Output: tensor of shape (..., inst, d_out)
    """

    A: nn.Parameter  # (n_inst, d_in, rank)
    B: nn.Parameter  # (n_inst, rank, d_out)

    def __init__(
        self,
        d_in: int = 768,
        d_out: int = 768,
        rank: int = 4,
        lora_alpha: float = 32,
        n_inst: int | None = None,
        dtype: t.dtype | None = None,
    ):
        """
        Initialize the weights of the LoRA block.
        - The A block should be initialized with kaiming uniform with a=sqrt(5)
        - The B block should be initialized with zeros.
        """
        super().__init__()
        self.rank = rank
        self.d_in = d_in
        self.d_out = d_out
        self.n_inst = 1 if n_inst is None else n_inst
        self.lora_alpha = lora_alpha
        self.dtype = dtype

        # Define the model parameters here
        self.A = nn.Parameter(kaiming_uniform_(t.empty((self.n_inst, self.d_in, self.rank)), a=np.sqrt(5)))
        self.B = nn.Parameter(t.zeros((self.n_inst, self.rank, self.d_out)))

    def forward(self, x: Float[Tensor, "... inst d_in"]) -> Float[Tensor, "... inst d_out"]:
        """
        Computes the forward pass of the LoRA block f(x) = (x @ A) @ B * lora_alpha / rank
        Args:
            x: Tensor of shape (..., inst, d_in)
        Returns:
            out (..., inst, d_out) such that out[..., i, :] = (x[..., i] @ A[i]) @ B[i] * lora_alpha / rank
        """
        if x.dtype != self.dtype:
            x = x.to(self.dtype)
        assert x.shape[-2] == self.n_inst or x.shape[-2] == 1, (
            f"Expected inst dim {self.n_inst} or 1, got {x.shape[-2]}. (input shape was {x.shape=})"
        )

        tmp = einops.einsum(x, self.A, '... inst d_in, inst d_in rank -> ... inst rank')
        out = einops.einsum(tmp, self.B, '... inst rank, inst rank d_out -> ... inst d_out')

        return out * self.lora_alpha / self.rank


model = HookedTransformer.from_pretrained("pythia-14m")
tests_lora.testing_lora(Lora)

Loaded pretrained model pythia-14m into HookedTransformer
test_lora passed
All tests for `Lora` passed!


<details><summary>Solution</summary>

```python
class Lora(nn.Module):
    """
    Module that implements the basic LoRA block.
    - Input: tensor of shape (..., [inst], d_in) and returns a tensor of shape (..., inst, d_out).
    - Calculated intermediate activations of shape (..., inst, rank)
    - Output: tensor of shape (..., inst, d_out)
    """

    A: nn.Parameter  # (n_inst, d_in, rank)
    B: nn.Parameter  # (n_inst, rank, d_out)

    def __init__(
        self,
        d_in: int = 768,
        d_out: int = 768,
        rank: int = 4,
        lora_alpha: float = 32,
        n_inst: int | None = None,
        dtype: t.dtype | None = None,
    ):
        """
        Initialize the weights of the LoRA block.
        - The A block should be initialized with kaiming uniform with a=sqrt(5)
        - The B block should be initialized with zeros.
        """
        super().__init__()
        self.rank = rank
        self.d_in = d_in
        self.d_out = d_out
        self.n_inst = 1 if n_inst is None else n_inst
        self.lora_alpha = lora_alpha
        self.dtype = dtype

        # Define the model parameters here
        self.A = nn.Parameter(t.empty(self.n_inst, d_in, rank, dtype=dtype))
        self.B = nn.Parameter(t.zeros(self.n_inst, rank, d_out, dtype=dtype))

        nn.init.kaiming_uniform_(self.A, a=5**0.5)

    def forward(self, x: Float[Tensor, "... inst d_in"]) -> Float[Tensor, "... inst d_out"]:
        """
        Computes the forward pass of the LoRA block f(x) = (x @ A) @ B * lora_alpha / rank
        Args:
            x: Tensor of shape (..., inst, d_in)
        Returns:
            out (..., inst, d_out) such that out[..., i, :] = (x[..., i] @ A[i]) @ B[i] * lora_alpha / rank
        """
        if x.dtype != self.dtype:
            x = x.to(self.dtype)
        assert x.shape[-2] == self.n_inst or x.shape[-2] == 1, (
            f"Expected inst dim {self.n_inst} or 1, got {x.shape[-2]}. (input shape was {x.shape=})"
        )


        # force order of operations (x A) B
        tmp = einops.einsum(x, self.A, "... inst d_in, inst d_in rank -> ... inst rank")
        out = einops.einsum(tmp, self.B, "... inst rank, inst rank d_out -> ... inst d_out")

        return out * self.lora_alpha / self.rank


model = HookedTransformer.from_pretrained("pythia-14m")
tests_lora.testing_lora(Lora)
```
</details>

## Attention LoRA

Next, we want to add code that adds the Low-Rank Adaptator to the attention layers. The original implementation of LoRA adds low-rank adaptors across all matrices in the attention layers ($W_Q, W_K, W_V, W_O$), leaving the MLP layers untouched.

Due to the way TransformerLens is implemented, we need to add hooks that cache the inputs to the attention sublayers, and then separately add hooks to modify the output of the attention sublayers, by loading the cached input from earlier, running it through the adaptor, and then adding it back to the output.

That is, we need to:

- Store the input to attention $W_Q$, $W_K$, and $W_V$ sublayers (all three recieve the same input, `normalized`) as well as the input to the $W_O$ sublayer (`z`)
- Modify the output of the attention $W_Q$, $W_K$, $W_V$, and $W_O$ sublayers (`q`, `k`, `v`, `attn_out`) by
    - Running the cached input through the adaptor, and
    - Adding the output from the adaptor to the original output of the module, and returning the result instead.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/LoraTransformerHookDiagram.png" width="960|">

### Exercise - complete `LoraHooks`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 25-30 minutes on this exercise.
> ```
Now you'll implement the `LoraHooks` class. This class should define LoRA modules for the linear projections inside the attention layer of the transformer. 

The following methods have been implemented for you:

* The hook function `store_hook_attn_normalized` should cache the input to querys, keys, and values.
* The hook function `store_hook_z` should cache the input to $W_O$.
* The method `list_fwd_hooks` should return a list of hook_point names and functions to call for the forward pass of the model using LoRA.

You should
* Define `self.lora_q`, `self.lora_k`, `self.lora_v`, `self.lora_o` of appropriate sizes.
* Implement the `lora_hook_qkv` method to apply the LoRA modules to the input to the attention layer.
   - This function should check the hook location (`hook.name`) and apply the appropriate LoRA modules to the appropriate input.
   - Note that `normalized : Float[Tensor, "batch pos d_model"]` is the input to the attention layer, which we need to repeat for each head before passing to the LoRA modules.
* Implement the `lora_hook_out` method to apply the LoRA modules to the output of the attention layer.
   - Note that `transformer_lens` doesn't hook the output of $W_O$ *before* the heads are summed over. Lucky for us, LoRA performs a linear operation, so we can sum the output of the LoRA model for $W_0$ over each head, and then add to the output!
   For example, for two heads:
   
   $$ 
   (\tilde{W}^1_O + \tilde{W}^2_O)(x) = x(W^1_O + A^1 B^1 + W^2_O + A^2 B^2) = x(W^1_O + W^2_O) + ((xA^1)B^1 + (xA^2) B^2)
   $$

<details>
<summary>What's the deal with <code>n_qo_heads</code> and <code>n_kv_heads</code>?</summary>

**TL;DR:** All you need to know is use `n_qo_heads` for the number of query heads and output heads, and `n_kv_heads` for the number of key and value heads.

For `gpt2`, we have the same number of heads in each layer, and each head has linear projections 
* $W_Q : (n_{heads},d_{model}, d_{head})$
* $W_K : (n_{heads},d_{model}, d_{head})$
* $W_V : (n_{heads},d_{model}, d_{head})$
* $W_O : (n_{heads}, d_{head}, d_{model})$

This turns out to be costly on memory when using KV-caching, so a solution proposed was [**grouped-query attention**](https://arxiv.org/pdf/2305.13245),
where there are fewer key and value heads than query heads. The query heads are put into groups, and each group shares the same
key and value head.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/f0d17ee6b9eb89dd72b746c43852a2dcb1245733/img/grouped_query_attention.png" width="960|">

We can see this in the family of Llama models which makes use of this technique.

```python
model = transformer_lens.HookedTransformer.from_pretrained("meta-llama/Llama-3.2-1B")
print(f"{model.cfg.n_heads=}")
print(f"{model.cfg.n_key_value_heads=}")
print(f"Group size: {model.cfg.n_heads // model.cfg.n_key_value_heads}")
print(f"{(model.W_K[0][:4] == model.W_K[0][0]).all()=}") 
```

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
model.cfg.n_heads=32
model.cfg.n_key_value_heads=8
Group size: 4
(model.W_K[0][:4] == model.W_K[0][0]).all()=tensor(True, device='cuda:0')
</pre>

Note that transformer lens presents `W_K` and `W_V` as the same shape as `W_Q`, but this is a lie, they are only a repeated view of the true key and value matricies, which are smaller. We can see this by looking inside the attention layer: `_W_K` is the *real* weights, and `W_K` is a repeated view of it.

```python
print(f"{model.blocks[0].attn._W_K.shape=}")
print(f"{model.blocks[0].attn.W_K.shape=}")
```

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">
model.blocks[0].attn._W_K.shape=torch.Size([8, 2048, 64])
model.blocks[0].attn.W_K.shape=torch.Size([32, 2048, 64])
</pre>


</details>

In [52]:
class LoraHooks(nn.Module):
    """
    Defines the LoRA hooks needed for the Attention Layers of the transformer.
    (Could be modified to add LoRA to the MLP layers)
    """

    lora_q: Lora
    lora_k: Lora
    lora_v: Lora
    lora_o: Lora
    cache_qkv_in: Float[Tensor, "batch pos d_model"] = None
    cache_z: Float[Tensor, "batch pos n_heads d_head"] = None

    def __init__(
        self,
        layer_idx: int,
        cfg: HookedTransformerConfig,
        lora_alpha: float = 32,
        rank: int = 4,
        dtype: t.dtype = None,
    ):
        super().__init__()
        self.layer_idx = layer_idx
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.dtype = dtype
        self.cfg=cfg

        self.n_qo_heads = n_qo_heads = cfg.n_heads
        self.n_kv_heads = n_kv_heads = cfg.n_key_value_heads if cfg.n_key_value_heads is not None else cfg.n_heads
        d_model, d_head = cfg.d_model, cfg.d_head

        self.lora_q = Lora(
            d_in=d_model, d_out=d_head, 
            rank=self.rank, lora_alpha=self.lora_alpha, 
            n_inst=self.n_qo_heads, dtype=self.dtype
        )
        self.lora_k = Lora(
            d_in=d_model, d_out=d_head, 
            rank=self.rank, lora_alpha=self.lora_alpha, 
            n_inst=self.n_kv_heads, dtype=self.dtype
        )
        self.lora_v = Lora(
            d_in=d_model, d_out=d_head, 
            rank=self.rank, lora_alpha=self.lora_alpha, 
            n_inst=self.n_kv_heads, dtype=self.dtype
        )
        self.lora_o = Lora(
            d_in=d_head, d_out=d_model, 
            rank=self.rank, lora_alpha=self.lora_alpha, 
            n_inst=self.n_qo_heads, dtype=self.dtype
        )

    def store_hook_attn_normalized(self, normalized: Float[Tensor, "batch pos d_model"], hook: HookPoint) -> None:
        """
        Cache the input to query/key/value.
        """
        self.cache_qkv_in = normalized

    def store_hook_z(self, z: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint) -> None:
        """
        Cache the input to $W_O$.
        """
        self.cache_z = z

    def list_fwd_hooks(self) -> list[tuple[str, Callable]]:
        """
        Returns a list of hook_point names and functions to call for the forward pass of
        the model using LoRA.
        """
        fwd_hooks = []
        # Attention Hooks qkv
        fwd_hooks.append((f"blocks.{self.layer_idx}.ln1.hook_normalized", self.store_hook_attn_normalized))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_q", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_k", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_v", self.lora_hook_qkv))
        # Attention Hooks z/out
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_z", self.store_hook_z))
        fwd_hooks.append((f"blocks.{self.layer_idx}.hook_attn_out", self.lora_hook_out))

        return fwd_hooks

    def lora_hook_qkv(
        self, qkv_hook_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to query/key/value, based on the hook location.
        Args:
            hook_qkv_out: Float[Tensor, "batch pos n_heads d_head"]
                The original output from query/key/value.
            hook: HookPoint
        Returns:
            The original output from query/key/value, plus the output from the corresponding LoRA module.
        """
        loras = dict(q=self.lora_q, k=self.lora_k, v=self.lora_v)
        lora = loras[hook.name[-1]]

        lora_contrib = lora(einops.repeat(self.cache_qkv_in, 'batch pos d_model -> batch pos 1 d_model'))
        return qkv_hook_out + lora_contrib

    def lora_hook_out(
        self, attn_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to the output projection matrix W_O in the attention layer.
        The output of the LoRA module is computed per head, so we sum over heads before adding
        to the activation `attn_out`.

        Args:
            attn_out: Float[Tensor, "batch pos n_heads d_head"]
                The output from the attention layer.
            hook: HookPoint
        Returns:
            The original output from the attention layer, plus the output from the LoRA module.
        """
        lora_contrib_per_head = self.lora_o(self.cache_z)
        lora_contrib = lora_contrib_per_head.sum(dim=2)
        return attn_out + lora_contrib

In [53]:
tests_lora.testing_lora_hooks(LoraHooks)
tests_lora.testing_lora_hooks_qkv_dispatch_and_out(LoraHooks)
print("All tests for LoraHooks passed!")

Tests for `LoraHooks` fwd_hooks passed!
Tests for `LoraHooks` dispatch and outputs passed!
All tests for LoraHooks passed!


<details><summary>Solution</summary>

```python
class LoraHooks(nn.Module):
    """
    Defines the LoRA hooks needed for the Attention Layers of the transformer.
    (Could be modified to add LoRA to the MLP layers)
    """

    lora_q: Lora
    lora_k: Lora
    lora_v: Lora
    lora_o: Lora
    cache_qkv_in: Float[Tensor, "batch pos d_model"] = None
    cache_z: Float[Tensor, "batch pos n_heads d_head"] = None

    def __init__(
        self,
        layer_idx: int,
        cfg: HookedTransformerConfig,
        lora_alpha: float = 32,
        rank: int = 4,
        dtype: t.dtype = None,
    ):
        super().__init__()
        self.layer_idx = layer_idx
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.dtype = dtype

        self.n_qo_heads = n_qo_heads = cfg.n_heads
        self.n_kv_heads = n_kv_heads = cfg.n_key_value_heads if cfg.n_key_value_heads is not None else cfg.n_heads
        d_model, d_head = cfg.d_model, cfg.d_head

        self.lora_q = Lora(d_model, d_head, n_inst=n_qo_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)
        self.lora_k = Lora(d_model, d_head, n_inst=n_kv_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)
        self.lora_v = Lora(d_model, d_head, n_inst=n_kv_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)
        self.lora_o = Lora(d_head, d_model, n_inst=n_qo_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)

    def store_hook_attn_normalized(self, normalized: Float[Tensor, "batch pos d_model"], hook: HookPoint) -> None:
        """
        Cache the input to query/key/value.
        """
        self.cache_qkv_in = normalized

    def store_hook_z(self, z: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint) -> None:
        """
        Cache the input to $W_O$.
        """
        self.cache_z = z

    def list_fwd_hooks(self) -> list[tuple[str, Callable]]:
        """
        Returns a list of hook_point names and functions to call for the forward pass of
        the model using LoRA.
        """
        fwd_hooks = []
        # Attention Hooks qkv
        fwd_hooks.append((f"blocks.{self.layer_idx}.ln1.hook_normalized", self.store_hook_attn_normalized))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_q", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_k", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_v", self.lora_hook_qkv))
        # Attention Hooks z/out
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_z", self.store_hook_z))
        fwd_hooks.append((f"blocks.{self.layer_idx}.hook_attn_out", self.lora_hook_out))

        return fwd_hooks

    def lora_hook_qkv(
        self, qkv_hook_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to query/key/value, based on the hook location.
        Args:
            hook_qkv_out: Float[Tensor, "batch pos n_heads d_head"]
                The original output from query/key/value.
            hook: HookPoint
        Returns:
            The original output from query/key/value, plus the output from the corresponding LoRA module.
        """

        hook_location = hook.name.split(".")[-1]

        qkv_in = self.cache_qkv_in
        qkv_in_repeated = einops.repeat(qkv_in, "batch pos d_model -> batch pos n_inst d_model", n_inst=1)

        if hook_location == "hook_q":
            return qkv_hook_out + self.lora_q(qkv_in_repeated)
        elif hook_location == "hook_k":
            return qkv_hook_out + self.lora_k(qkv_in_repeated)
        elif hook_location == "hook_v":
            return qkv_hook_out + self.lora_v(qkv_in_repeated)
        else:
            raise ValueError(f"Invalid hook location: {hook_location}")

    def lora_hook_out(
        self, attn_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to the output projection matrix W_O in the attention layer.
        The output of the LoRA module is computed per head, so we sum over heads before adding
        to the activation `attn_out`.

        Args:
            attn_out: Float[Tensor, "batch pos n_heads d_head"]
                The output from the attention layer.
            hook: HookPoint
        Returns:
            The original output from the attention layer, plus the output from the LoRA module.
        """

        lora_result = self.lora_o(self.cache_z)
        lora_attn_out = einops.einsum(lora_result, "... n_heads d_model -> ... d_model")
        return attn_out + lora_attn_out
```
</details>

## Training with LoRA

We can now define a modified form on the `HookedTransformerWithValueHead` class that includes a LoRA module attached to every attention layer.
This means when we train the model, we no longer need an additional reference model, but can simply turn the LoRA modules on and off. With the LoRA modules disabled, the model will act like the base model, as the original parameters of the model are not modified.

### Exercise - complete `TransformerWithValueHeadLora`

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 25-30 minutes on this exercise.
> ```
Now you'll implement the `LoraHooks` class. This class should define LoRA modules for the linear projections inside the attention layer of the transformer. 


You should
* Define `setup_lora` which will 
    - define `self.lora` as a `nn.ModuleList` of `LoraHooks` for each layer. 
    - defines `self.lora_fwd_hooks` as a list of all the forward hooks for the LoRA modules. 
    - You shouldmake use of the `list_fwd_hooks()` method we defined for you earlier.

* Define `forward_with_value_head` which will use the `fwd_hooks` property to forward the model with the LoRA modules enabled. 
    - Make use of the `with.self.hooks(fwd_hooks=self.fwd_hooks)` context manager.
    - This function is quite simple, only a few lines.
* Define `generate` to override the `generate` method in the parent class to use the LoRA hooks.
    - Also simple, should be a similar implementation to `forward_with_value_head`.

In [55]:
class TransformerWithValueHeadLora(HookedTransformerWithValueHead):
    lora: nn.ModuleList
    lora_fwd_hooks: list[tuple[str, Callable]]
    dtype: t.dtype
    device: t.device
    use_value_head: bool

    def base_model_params(self):
        return (p for name, p in self.named_parameters() if "value_head" not in name and "lora" not in name)

    def lora_params(self):
        return self.lora.parameters()

    # we use these for compatibility with get_optimizer_and_scheduler
    def get_base_model_trainable_params(self):
        return self.lora_params()

    def get_value_head_params(self):
        return (p for name, p in self.named_parameters() if "value_head" in name)

    @classmethod
    def from_pretrained(cls, *args, lora_alpha: float = 32, rank: int = 4, **kwargs):
        model = super(TransformerWithValueHeadLora, cls).from_pretrained(*args, **kwargs)
        model.setup_lora(lora_alpha=lora_alpha, rank=rank, **kwargs)

        for param in model.base_model_params():
            param.requires_grad = False

        return model

    def setup_lora(self, lora_alpha: float = 32, rank: int = 4, **kwargs):
        """
        Initializes LoRA (Low-Rank Adaptation) for all attention layers in the transformer.

        Steps of this function are:
           - Creates a LoraHooks module for each transformer layer
           - Creates the list of forward hooks for all layers
        """

        self.lora = nn.ModuleList((
            LoraHooks(layer_idx, self.cfg, lora_alpha=lora_alpha, rank=rank)
            for layer_idx in range(self.cfg.n_layers)
        )).to(device)
        self.lora_fwd_hooks = []
        for lora_module in self.lora:
            self.lora_fwd_hooks += lora_module.list_fwd_hooks()

    @property
    def fwd_hooks(self):
        return self.lora_fwd_hooks + [self.value_head_hook]

    def forward_with_value_head(
        self, tokens: Int[Tensor, "batch seq"]
    ) -> tuple[Float[Tensor, "batch seq d_vocab"], Float[Tensor, "batch seq"]]:
        """
        Forward pass with LoRA enabled, including the value head outputs.

        Args:
            tokens: Int[Tensor, "batch seq"]
                The input tokens to the transformer.
        Returns:
            logits: Float[Tensor, "batch seq d_vocab"]
                The logits of the transformer.
            value: Float[Tensor, "batch seq"]
                The value head outputs for each token.
        """
        with self.hooks(fwd_hooks=self.fwd_hooks):
            logits = self.forward(tokens)
        value = self.value_head_output
        return logits, value

    @t.no_grad()
    def generate(self, tokens: Int[Tensor, "batch seq"], **kwargs) -> Int[Tensor, "batch seq"]:
        """
        We override the generate method to use the LoRA hooks applied so that we don't need to update the previous training code.
        This function should call generate on the parent class (HookedTransformer), but with the LoRA hooks applied.
        We don't need to return the value head outputs during generation.

        Args:
            tokens: Int[Tensor, "batch seq"]
                The input tokens to the transformer.
            **kwargs:
                Additional keyword arguments to pass to the base class generate method.
        Returns:
            gen_tokens: Int[Tensor, "batch gen_len"]
                The generated tokens.
        """
        with self.hooks(fwd_hooks=self.fwd_hooks):
            return super().generate(tokens, **kwargs)


model = TransformerWithValueHeadLora.from_pretrained("pythia-14m").to(device)
tests_lora.test_lora_fwd_hooks_list(model)
tests_lora.test_lora_model_forward_methods(model)
print("All tests for TransformerWithValueHeadLora passed!")

Loaded pretrained model pythia-14m into HookedTransformer
Moving model to device:  cuda
testing lora fwd hooks list passed!


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

All tests for TransformerWithValueHeadLora passed!


<details><summary>Solution</summary>

```python
class TransformerWithValueHeadLora(HookedTransformerWithValueHead):
    lora: nn.ModuleList
    lora_fwd_hooks: list[tuple[str, Callable]]
    dtype: t.dtype
    device: t.device
    use_value_head: bool

    def base_model_params(self):
        return (p for name, p in self.named_parameters() if "value_head" not in name and "lora" not in name)

    def lora_params(self):
        return self.lora.parameters()

    # we use these for compatibility with get_optimizer_and_scheduler
    def get_base_model_trainable_params(self):
        return self.lora_params()

    def get_value_head_params(self):
        return (p for name, p in self.named_parameters() if "value_head" in name)

    @classmethod
    def from_pretrained(cls, *args, lora_alpha: float = 32, rank: int = 4, **kwargs):
        model = super(TransformerWithValueHeadLora, cls).from_pretrained(*args, **kwargs)
        model.setup_lora(lora_alpha=lora_alpha, rank=rank, **kwargs)

        for param in model.base_model_params():
            param.requires_grad = False

        return model

    def setup_lora(self, lora_alpha: float = 32, rank: int = 4, **kwargs):
        """
        Initializes LoRA (Low-Rank Adaptation) for all attention layers in the transformer.

        Steps of this function are:
           - Creates a LoraHooks module for each transformer layer
           - Creates the list of forward hooks for all layers
        """

        self.lora = nn.ModuleList(
            [LoraHooks(layer_idx, self.cfg, lora_alpha, rank) for layer_idx in range(len(self.blocks))]
        ).to(device)

        # create list of all hooks for all layers
        self.lora_fwd_hooks = []
        for layer_idx in range(len(self.blocks)):
            self.lora_fwd_hooks.extend(self.lora[layer_idx].list_fwd_hooks())

    @property
    def fwd_hooks(self):
        return self.lora_fwd_hooks + [self.value_head_hook]

    def forward_with_value_head(
        self, tokens: Int[Tensor, "batch seq"]
    ) -> tuple[Float[Tensor, "batch seq d_vocab"], Float[Tensor, "batch seq"]]:
        """
        Forward pass with LoRA enabled, including the value head outputs.

        Args:
            tokens: Int[Tensor, "batch seq"]
                The input tokens to the transformer.
        Returns:
            logits: Float[Tensor, "batch seq d_vocab"]
                The logits of the transformer.
            value: Float[Tensor, "batch seq"]
                The value head outputs for each token.
        """

        with self.hooks(fwd_hooks=self.fwd_hooks):
            logits = self.forward(tokens)
        value = self.value_head_output
        return logits, value

    @t.no_grad()
    def generate(self, tokens: Int[Tensor, "batch seq"], **kwargs) -> Int[Tensor, "batch seq"]:
        """
        We override the generate method to use the LoRA hooks applied so that we don't need to update the previous training code.
        This function should call generate on the parent class (HookedTransformer), but with the LoRA hooks applied.
        We don't need to return the value head outputs during generation.

        Args:
            tokens: Int[Tensor, "batch seq"]
                The input tokens to the transformer.
            **kwargs:
                Additional keyword arguments to pass to the base class generate method.
        Returns:
            gen_tokens: Int[Tensor, "batch gen_len"]
                The generated tokens.
        """

        with self.hooks(fwd_hooks=self.lora_fwd_hooks):
            gen_tokens = super().generate(tokens, **kwargs)
        return gen_tokens


model = TransformerWithValueHeadLora.from_pretrained("pythia-14m").to(device)
tests_lora.test_lora_fwd_hooks_list(model)
tests_lora.test_lora_model_forward_methods(model)
print("All tests for TransformerWithValueHeadLora passed!")
```
</details>

Since we still need the reference model, and since we don't modify the base model weights directly, we can load the base model once and apply LoRA via forward hooks for training, while also using the same base as the frozen reference policy (without LoRA hooks) for KL.

<details>
<summary>Why do we only add LoRA to the attention layers?</summary>

The original LoRA paper adds adapters to attention projections. More recent work often also adds them to MLP layers, or even only to MLP. As a bonus exercise, you can try adding LoRA to the MLP layers too!

</details>

In [56]:
@dataclass
class RLHFArgsLora(RLHFArgs):
    lora_rank: int = 4
    lora_alpha: float = 32
    dtype: t.dtype = None


class RLHFTrainerLora(RLHFTrainer):
    model: TransformerWithValueHeadLora
    memory: ReplayMemory

    def __init__(self, args: RLHFArgsLora):
        """
        Method that now loads the reference model and the lora_model.
        """
        t.manual_seed(args.seed)
        self.args = args
        self.run_name = f"{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        self.model = TransformerWithValueHeadLora.from_pretrained(
            args.base_model, lora_alpha=args.lora_alpha, rank=args.lora_rank
        )
        self.model.to(device).train()
        self.ref_model = self.model  # no need for seperate reference model!

        self.optimizer, self.scheduler = get_optimizer_and_scheduler(self.args, self.model)
        self.prefix_len = len(self.model.to_str_tokens(self.args.prefix, prepend_bos=self.args.prepend_bos))

In [57]:
print("Training LoRA model RLHF (example setup)")
lora_args = RLHFArgsLora(
    use_wandb=False,
    kl_coef=0.0,
    total_phases=2,
    warmup_steps=0,
    reward_fn=reward_fn_char_count,
    base_lr=1e-3,
    batch_size=8,
    num_minibatches=2,
    gen_len=8,
)
lora_trainer = RLHFTrainerLora(lora_args)
lora_trainer.train()  # Uncomment to run a tiny smoke test

Training LoRA model RLHF (example setup)
Loaded pretrained model gpt2-medium into HookedTransformer
Moving model to device:  cuda


  0%|          | 0/2 [00:00<?, ?it/s]/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:288: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  assert torch.tensor(shape).prod().item() == index_tensor[idx].numel(), \
/root/miniconda3/envs/arena-env/lib/python3.11/site-packages/eindex/indexing.py:292: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch

Phase 001/2, Mean reward: 0.3750
┌──────────┬────────────────┬─────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                      │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────┤
│        0 │         -18.96 │ '<|endoftext|>This is a list of all the events where the'   │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────┤
│        1 │          -3.2  │ '<|endoftext|>This is a rush transcript. Copy may not be'   │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────┤
│        1 │         -20.75 │ '<|endoftext|>This is a great game that I would recommend.' │
└──────────┴────────────────┴─────────────────────────────────────────────────────────────┘



Loss: -0.0251:  50%|█████     | 1/2 [00:01<00:01,  1.14s/it]

Phase 002/2, Mean reward: 0.1250
┌──────────┬────────────────┬─────────────────────────────────────────────────────────────────┐
│   Reward │   Ref logprobs │ Sample                                                          │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────────┤
│        0 │         -25.07 │ '<|endoftext|>This is a beautiful piece that I love! This'      │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────────┤
│        0 │         -24.21 │ '<|endoftext|>This is a pretty amazing product and I have been' │
├──────────┼────────────────┼─────────────────────────────────────────────────────────────────┤
│        1 │         -21.04 │ '<|endoftext|>This is a great app. It lets you know'            │
└──────────┴────────────────┴─────────────────────────────────────────────────────────────────┘



Loss: -0.0463: 100%|██████████| 2/2 [00:02<00:00,  1.09s/it]


# 3️⃣ GRPO LoRA

> ##### Learning Objectives
>
> - Understand and implement GRPO
> - Use GRPO + LoRA together to finetune a model.

## Group Relative Policy Optimization (🚧 Under construction 🚧)

GRPO is a variant of PPO specialised for doing RLHF on LLMs.
It was first described in Apr 2024 for use for fine-tuning DeepSeek to achieve better performance on tasks that require reasoning, by reinforcing rollouts that lead to correct answers.

The main differences between PPO and GRPO is that:
* PPO uses a critic head to estimate the baseline. GRPO removes the critic entirely, and instead performs many rollouts, and uses the average reward over those rollouts as a baseline function.
* PPO computes the advantages using GAE. GRPO simply uses the normalized rewards for the set of rollouts as the advantages.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/grpo.png" width="960|">

Letting $o_{1:T}$ be a sequence of tokens, the joy function (loss function but we maximize) is given as 

$$
J_{\text{GRPO}}(\theta)
= \widehat{\mathbb{E}} \left[ \frac{1}{T} \sum_{t=1}^{T}
\left(
\min\Big[ \rho_\theta(o_t \mid o_{<t}) \, \hat{A}_t, \; \text{clip}(\rho_\theta(o_t \mid o_{<t}), 1-\epsilon, 1+\epsilon)\,\hat{A}_t \Big]
\; - \beta \, D_{\mathrm{KL}}\!\left[\pi_\theta \,\|\, \pi_{\text{ref}}\right]
\right) \right].
$$
where
$$
\rho_\theta(o_t \mid o_{<t}) = \frac{\pi_\theta(o_t \mid o_{<t})}{\pi_{\theta_{\text{old}}}(o_t \mid o_{<t})}
$$
is the probability ratio of the new policy to the old policy.

The advantages are now just the normalized rewards:
$$
\hat{A}_t = (r_t - \text{mean}(\mathbf{r})) / \text{std}(\mathbf{r})
$$
where $\mu_r$ is the mean of the rewards vector, and $\sigma_r$ is the standard deviation of the rewards vector.

For the moment, we just superclass the existing `TransformerWithValueHeadLora` class and skip the value head. This is hacky, but it's a quick way to get the code working.

In [ ]:
class TransformerWithLora(TransformerWithValueHeadLora):
    "We don't need the value head for training with GRPO"

    lora: nn.ModuleList
    lora_fwd_hooks: list[tuple[str, Callable]]
    dtype: t.dtype
    device: t.device

    def get_value_head_params(self):
        return iter([])  # no value head parameters

    @classmethod
    def from_pretrained(cls, *args, lora_alpha: float = 32, rank: int = 4, **kwargs):
        model = super(TransformerWithLora, cls).from_pretrained(*args, use_value_head=False, **kwargs)
        model.value_head_output = None
        return model

    @property
    def fwd_hooks(self):
        return self.lora_fwd_hooks  # no value head hook

    def forward_with_value_head(
        self, tokens: Int[Tensor, "batch seq"]
    ) -> tuple[Float[Tensor, "batch seq d_vocab"], Float[Tensor, "batch seq"]]:
        """
        Forward pass with LoRA enabled, but value head is not used.
        """
        logits, value = super().forward_with_value_head(tokens)
        assert value is None, "Value head got run somehow?"
        return logits

In GRPO-style training we optimize only the policy objective plus regularizers, without a value head or critic loss. This simplifies the architecture when your reward is available at the sequence level and you propagate it per generated token.

* We re-use the optimizer and scheduler helpers. 
* In the rollout, we compute rewards per sample, optionally normalize them, and use them as advantages for all generated positions. 
* In learning, we maximize the clipped objective with an entropy bonus, and subtract the KL penalty computed against the frozen reference model.

### Exercise: Construct GRPO trainer

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 40 minutes on this exercise.
> ```

Construct a GRPO trainer class that inherits from `RLHFTrainer` and overrides the `rollout_phase` and `learning_phase` methods.

We recommend copying the solution for `RLHFTrainer` for PPO, and then modifying it to work for GRPO.
This will msotly involve chopping parts out, or replacing parts (e.g. calculation of the advantage.)

You could also redefine `calc_value_function_loss` and `compute_advantages`, and then try to use `RLHFTrainer` as is.

The rough changes should be
* Drop the value head and the associated critic loss
* Use normalized rewards as the advantages. As advantages are of shape `(minibatch, seq_len)`, and rewards are of shape `(minibatch,)`, we need to deal with this somehow. Looking at Section 4 in the GRPO paper:
    - **4.1.2 Outcome Supervision**: Treat each advantage as the reward we get at the end of the sequence. 
        - Essentially we repeat the rewards for each token in the sequence. **We use this approach.**
    - **4.1.2 Process Supervision**: Query the reward function for every prefix directly, and the advantage becomes the returns
    $$
    \hat{A}_t = \sum_{t'=t}^{T} \tilde{\mathbf{r}}_{t'}
    $$
    where $\tilde{\mathbf{r}} = \mathbf{r} - \text{mean}(\mathbf{r}) / \text{std}(\mathbf{r})$ is the normalized rewards.
    This can get expensive if done on a per-token basis, but for large CoT's the generation is broken into "thoughts" rather than tokens (e.g. sentences?).


<details>
<summary>Hint for `compute_rlhf_objective`</summary>

If you modify `TransformerWithLora` to return a tensor of zeros of appropriate size, and redefine `calc_value_function_loss` to just return zero, you should be able to use `compute_rlhf_objective` as is.
We don't do that here, but just redefine the function and remove parts.

</details>

In [ ]:
@dataclass
class GrpoArgs(RLHFArgs):
    lora_rank: int = 4
    lora_alpha: float = 32


class GrpoTrainer(RLHFTrainer):
    model: TransformerWithLora
    memory: ReplayMemory

    def __init__(self, args: RLHFArgs):
        # duplicates code from RLHFTrainerLora
        t.manual_seed(args.seed)
        self.args = args
        self.run_name = f"{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        self.model = TransformerWithLora.from_pretrained(args.base_model).to(device).train()
        self.ref_model = self.model
        self.optimizer, self.scheduler = get_optimizer_and_scheduler(self.args, self.model)
        self.prefix_len = len(self.model.to_str_tokens(self.args.prefix, prepend_bos=self.args.prepend_bos))

    def compute_rlhf_objective(self, minibatch: ReplayMinibatch):
        raise NotImplementedError()

    def rollout_phase(self) -> ReplayMemory:
        raise NotImplementedError()

In [ ]:
print("Training GRPO model (example setup)")
grpo_args = GrpoArgs(
    use_wandb=False,
    kl_coef=2.5,
    total_phases=30,
    warmup_steps=0,
    reward_fn=reward_fn_char_count,
    base_lr=1e-3,
    # batch_size=8,
    # num_minibatches=2,
    gen_len=16,
)
grpo_trainer = GrpoTrainer(grpo_args)
grpo_trainer.train()  # Uncomment to run a tiny smoke test

<details><summary>Solution</summary>

```python
@dataclass
class GrpoArgs(RLHFArgs):
    lora_rank: int = 4
    lora_alpha: float = 32


class GrpoTrainer(RLHFTrainer):
    model: TransformerWithLora
    memory: ReplayMemory

    def __init__(self, args: RLHFArgs):
        # duplicates code from RLHFTrainerLora
        t.manual_seed(args.seed)
        self.args = args
        self.run_name = f"{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        self.model = TransformerWithLora.from_pretrained(args.base_model).to(device).train()
        self.ref_model = self.model
        self.optimizer, self.scheduler = get_optimizer_and_scheduler(self.args, self.model)
        self.prefix_len = len(self.model.to_str_tokens(self.args.prefix, prepend_bos=self.args.prepend_bos))

    def compute_rlhf_objective(self, minibatch: ReplayMinibatch):

        gen_len_slice = slice(-self.args.gen_len - 1, -1)

        logits, values = self.model.forward_with_value_head(minibatch.sample_ids)

        logprobs = get_logprobs(logits, minibatch.sample_ids, self.prefix_len)

        clipped_surrogate_objective = calc_clipped_surrogate_objective(
            logprobs,
            minibatch.logprobs,
            minibatch.advantages,
            self.args.clip_coef,
            self.args.gen_len,
        )
        entropy_bonus = calc_entropy_bonus(logits[:, gen_len_slice], self.args.ent_coef, self.args.gen_len)
        kl_penalty = calc_kl_penalty(
            logits[:, gen_len_slice],
            minibatch.ref_logits[:, gen_len_slice],
            self.args.kl_coef,
            self.args.gen_len,
        )

        ppo_objective_fn = clipped_surrogate_objective + entropy_bonus
        total_objective_function = ppo_objective_fn - kl_penalty

        if self.args.use_wandb:
            with t.inference_mode():
                logratio = logprobs - minibatch.logprobs
                ratio = logratio.exp()
                clipfracs = [((ratio - 1.0).abs() > self.args.clip_coef).float().mean().item()]
            wandb.log(
                dict(
                    total_steps=self.step,
                    lr=self.scheduler.get_last_lr()[0],
                    clipped_surrogate_objective=clipped_surrogate_objective.item(),
                    clipfrac=np.mean(clipfracs),
                    entropy_bonus=entropy_bonus.item(),
                    kl_penalty=kl_penalty.item(),
                ),
                step=self.step,
            )

        return total_objective_function

    def rollout_phase(self) -> ReplayMemory:

        sample_ids, samples = get_samples(
            self.model,
            prompt=self.args.prefix,
            batch_size=self.args.batch_size,
            gen_len=self.args.gen_len,
            temperature=self.args.temperature,
            top_k=self.args.top_k,
            prepend_bos=self.args.prepend_bos,
        )

        with t.inference_mode():
            logits, values = self.model.forward_with_value_head(sample_ids)
            ref_logits = self.ref_model(sample_ids)

        logprobs = get_logprobs(logits, sample_ids, self.prefix_len)

        rewards = self.args.reward_fn(samples)
        rewards_mean = rewards.mean().item()
        rewards_normed = normalize_reward(rewards) if self.args.normalize_reward else rewards

        advantages = rewards_normed

        if self.args.use_wandb:
            wandb.log({"mean_reward": rewards_mean}, step=self.step)

        n_log_samples = min(5, self.args.batch_size)
        ref_logprobs = get_logprobs(ref_logits[:n_log_samples], sample_ids[:n_log_samples], self.prefix_len).sum(-1)
        headers = ["Reward", "Ref logprobs", "Sample"]
        table_data = [[str(int(r)), f"{lp:.2f}", repr(s)] for r, lp, s in zip(rewards.tolist(), ref_logprobs, samples)]
        table = tabulate(table_data, headers, tablefmt="simple_grid", maxcolwidths=[None, None, 90])
        print(f"Phase {self.phase + 1:03}/{self.args.total_phases:03}, Mean reward: {rewards_mean:.4f}\n{table}\n")

        values = einops.repeat(advantages, "b -> b g", g=sample_ids.shape[1])
        advantages = einops.repeat(advantages, "b -> b g", g=logprobs.shape[1])
        return ReplayMemory(
            args=self.args,
            sample_ids=sample_ids,
            logprobs=logprobs,
            advantages=advantages,
            values=values,
            ref_logits=ref_logits,
        )
```
</details>

# ☆ Bonus

> ##### Learning Objectives
>
> - Improve your RLHF implementation via techniques like differential learning rates, frozen layers, or adaptive KL penalties
> - Perform some exploratory mechanistic interpretability on RLHF'd models
> - Learn about the trlX library, which is designed to train transformers via RLHF in a way which abstracts away many of the low-level details

## Extensions of today's RLHF exercises

### Large models

We're already working with `gpt2-medium` which is considerably larger than most of the models you worked with in most of the transformers & interpretability material. Can you go even larger, e.g. `gpt2-xl` or more?

See [this page](https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html) for a table of model properties, for all models currently supported by TransformerLens. Note that if you use different model classes then you might need to change some parts of your code (e.g. if the name of the hook point where you added the value head happens to be different). You might also need to make other adjustments e.g. a smaller batch size (or a larger number of minibatches per batch, which is equivalent to smaller minibatch sizes).

### Differential Learning Rates / Frozen Layers

When doing any kind of finetuning, it's common practice to either freeze earlier layers or have a smaller learning rate for them. You may have seen this in the feature extraction with ResNet34 exercises in the first week. In the exercises here we've trained all layers of the model equally, but you might want to play around with differential learning rates.

Note that you can accomplish this using parameter groups - we already used parameter groups above to have a different learning rate for our base model and value head. It should be relatively straightforward to extend this to splitting parameters over different layers into different groups (hint - you can use `itertools.chain` to convert several iterables into a single iterable).

You can also try entirely freezing earlier layers - this might also reduce your memory usage, and allow you to train larger models without getting cuda errors.

### Hyperparameter sweeps

You can do this to find the best possible hyperparamters for your RLHF training. Don't just measure on reward, can you use some combination of reward and avg kl diff to create a better metric? Can you use wandb's built-in [Bayesian search methods](https://docs.wandb.ai/guides/sweeps/sweep-config-keys#bayesian-search) to more effectively sweep?

Note - don't forget **temperature** when it comes to hyperparameter tuning. Temperature has an important effect on how the model learns, e.g. if the temperature is too high then the model will produce very high-variance outputs which will have very high KL with the reference distribution, and it'll be more likely to collapse into some incoherent mode.

### Adaptive KL penalty

The KL divergence penalty coefficient can be modified adaptively based on the KL divergence between the current policy and the previous policy. If the KL divergence is outside a predefined target range, we can adjust the penalty coefficient to bring it closer to the target range. Here is an example implementation:

```python
class AdaptiveKLController:
    def __init__(self, init_kl_coef, hparams):
        self.value = init_kl_coef
        self.hparams = hparams

    def update(self, current, n_steps):
        target = self.hparams.target
        proportional_error = np.clip(current / target - 1, -0.2, 0.2)
        mult = 1 + proportional_error * n_steps / self.hparams.horizon
        self.value *= mult
```

### TRL / trlX

We've been focusing on building RLHF from the ground up, but there are several libraries which exist to abstract away manuy of the low-level implementation details we had to wrestle with. One of the best-known is TRL (Transformer Reinforcement Learning). The main docs page can be found [here](https://huggingface.co/docs/trl/index), and [this page](https://huggingface.co/docs/trl/quickstart) gives a quickstart guide. You may find it much easier to use this library than to implement everything yourself!

Read their documentation pages, and see what techniques they use to make RLHF more effective. Are there any that we haven't implemented here? Can you implement them yourself?

You might also be interested in trlX, an expanded fork of TRL built by CarperAI to handle larger models for online and offline training (although their APIs are pretty similar).

### Learn a human preference reward model

We've been working with a pre-supplied reward function, but you can try and train your own!

We'll give some brief points of guidance here, for the task of training a reward function on the **summarization task**. Note that these instructions have been provided externally, so they've not yet been tested and might not work particularly well.

1. Get a supervised baseline
    * [Here](https://zenodo.org/records/1168855) is a link to download the dataset for the TL;DR challenge containing posts from the Reddit corpus. Each post contains keys `content` and `summary` which are the original post and the human-written summary respectively.
    * You should throw out all summaries shorter than 24 tokens or longer than 48 tokens (to diminish the effects of length on quality); and choose a random subset of ~100k summaries to train on.
    * Run training to maximize the log-likelihood of these summaries.
2. Get reward model by training supervised baseline on human feedback
    * Download comparison data with the code `azcopy copy "https://openaipublic.blob.core.windows.net/summarize-from-feedback/dataset/*" . --recursive`
    * Modify GPT-2 architecture by adding a randomly-initialized **reward head** at the end of your model.
        * Architecturally this is similar to the value head from earlier, but it's not the same thing - here we're trying to learn what the human reward will be; we're not doing RL yet.
    * Train your model (starting with base model given by supervised baseline weights, and reward head randomly initialized) to minimize `loss = log(sigmoid(reward_model(summary_0) - reward_model(summary_1)))`, `summary_0` is preferred by a human labeler (this data should be in the comparison data you downloaded).
    * You should normalize reward model outputs, like we normalized rewards in RLHF in previous exercises.
3. Fine-tune supervised baseline using PPO with reward model.
    * For these exercises we suggest using a larger model, ideally GPT2-Large or bigger. Remember you can freeze weights! Regardless, this will still take longer to train than your previous models.

### Interp on RLHF'd models

Currently, very little mechanistic interpretability research ahs focused on RLHF'd models. In [this blog post](https://blog.eleuther.ai/trlx-exploratory-analysis/), Curt Tigges walks through an example of how we can use mech interp to analyze a model which has been finetuned with a sentiment based reward function using trlX.

The flavour of the actual mech interp done here is very similar to the indirect object identification exercises you might have done during the transformers & interp week. If you didn't do these exercises, we recommend you do them before diving deep into this material.

Lastly, here's a [Google Doc](https://docs.google.com/document/d/1eUdvlJNqY9X0NAw9UUseZz6dFyRklCcOHQy8x3CbcBk/edit?usp=sharing) brainstorming some ideas for RLHF interpretability. You might find some ideas there (although most of these will be pretty vague goals so possibly too ambitious for a bonus exercise or 1-week project).

## Suggested paper replications

As well as the papers in this section, you might be interested in browsing this [GitHub repo](https://github.com/opendilab/awesome-RLHF), which contains links to a large number of RLHF-related papers.

### [Deep Reinforcement Learning from Human Preferences](https://arxiv.org/abs/1706.03741)

This was the seminal paper in RLHF. They applied it to the domain of tasks like MuJoCo (which you might already have worked with during your PPO day). Can you set up a reward function and an interface which allows you to choose between two different sets of trajectories, and learn a reward function to maximize?

Some more technical details here - the authors train the reward function at the same time as they train the model. In other words, after a certain number of iterations of (rollout phase, learning phase), they add a third reward model learning phase, where the current policy generates many pairs of trajectories of some fixed timestep and the human rater chooses which one is best. They famously trained the Hopper agent to perform repeated backflips using just 900 queries.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/hopper-backflip.png" width="700">

[Here](https://drive.google.com/drive/folders/0BwcFziBYuA8RM2NTdllSNVNTWTg?resourcekey=0-w4PuSuFvi3odgQXdBDPQ0g) is the link mentioned in the image caption.

Note - we strongly recommend doing the PPO exercises on MuJoCo before attempting this replication. We also recommend using Colab, since MuJoCo is notoriously difficult to install all the dependencies for!

### [Recursively Summarizing Books with Human Feedback](https://arxiv.org/abs/2109.10862)

A major challenge for scaling ML is training models to perform tasks that are very difficult or time-consuming for humans to evaluate. To test scalable alignment techniques, the authors trained a model to summarize entire books, by first summarizing small sections of a book, then summarizing those summaries into a higher-level summary, and so on. A demonstration can be found [here](https://openai.com/research/summarizing-books). There is also a [repository](https://github.com/openai/summarize-from-feedback) containing code to run their models, including the supervised baseline, the trained reward model, and the RL fine tuned policy.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/alice.png" width="500">

You may also wish to do this in a less directed way - see the bonus exercise “Learn a human preference reward model” above.

### Extentions for LoRA

### Extend `Lora` to MLP layers (optional)

<!-- > ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: ⚪⚪⚪⚪⚪
> 
> You should spend up to 20 minutes on this exercise.
> ``` -->

This is a pretty finicky exercise, and mostly involves looking up the various locations you can add hook functions to.
Modify `HookedTransformer` to add extra LoRA layers across the MLP project up and project down layers.


<details>
<summary>My solution doesn't work for Llama models!</summary>
For non-GPT2 models this is even more annoying, as the architecture is different and involves Gated Linear Units (GLUs).
We leave this to you to work out.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/ArenaLoraAttnMlpHooksDiagram.png" width="1624">

</details>

### Mixed Precision (optional)

We can squeeze even more out of the training process by training in mixed precision. We can load the models in bfloat16, and train with that instead of float32.

Due to <a href="https://github.com/TransformerLensOrg/TransformerLens/issues/104#issuecomment-1597178527">this issue on mixed precision</a>, TransformerLens uses float32 for LayerNorms even if the rest of the model is using bfloat16 or float16. This means the intermediate activations are in float32, even though the weights are in bfloat16.


### complete `LoraMixedPrecision`
<!-- 
> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: ⚪⚪⚪⚪⚪
> 
> You should spend up to 10 minutes on this exercise.
> ``` -->

To handle this, you should modify hooks to:

- store the original `dtype` of the input
- use the passed in `dtype` to convert the input
- do the normal computations
- convert the output back to the original `dtype`

You can use `super().lora_hook_qkv(act, hook)` to call the original `lora_hook_qkv` method, and then wrap this
in the appropriate code to cast the types back and forth.

```python
class LoraHooksMixedPrecision(LoraHooks):
    """
    Defines the LoRA hooks needed for the Attention layer of the transformer, but allow for mixed precision.
    """
    
    def lora_hook_qkv(
        self, 
        resid_pre_normed: Float[Tensor, "batch pos d_model"], 
        hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        
        raise NotImplementedError()

    def lora_hook_out(
        self, attn_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        raise NotImplementedError()
   
```

<details>
<summary>Solution</summary>

```python
class LoraHooksMixedPrecision(LoraHooks):
    """
    Defines the LoRA hooks needed for the Attention layer of the transformer, but allow for mixed precision.
    """
    
    def lora_hook_qkv(
        self, 
        resid_pre_normed: Float[Tensor, "batch pos d_model"], 
        hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        
        # EXERCISE
        # raise NotImplementedError()
        # END EXERCISE
        # SOLUTION
        hook_location = hook.name.split(".")[-1]
        
        orig_dtype = resid_pre_normed.dtype
        resid_pre_normed = resid_pre_normed.to(self.dtype)
      
        super().lora_hook_qkv(resid_pre_normed, hook)
        
        lora_qkv_out = lora_qkv_out.to(orig_dtype)
        return lora_qkv_out
        # END SOLUTION

    def lora_hook_out(
        self, attn_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        
        # EXERCISE
        # raise NotImplementedError()
        # END EXERCISE
        # SOLUTION
        orig_dtype = attn_out.dtype
        attn_out = attn_out.to(self.dtype)
        
        super().lora_hook_out(attn_out, hook)
        
        lora_attn_out = lora_attn_out.to(orig_dtype)
        return lora_attn_out
        # END SOLUTION
```

</details>